In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Project data directory
DATA_DIR = Path("../data/ehr")

# Check whether the directory exists
print("EHR directory exists:", DATA_DIR.exists())

# Display all CSV files
csv_files = sorted(DATA_DIR.glob("*.csv"))

print("Number of CSV files:", len(csv_files))

for file in csv_files:
    print(file.name)

In [ ]:
# Load each CSV into a dictionary of DataFrames
ehr_tables = {}

for file in csv_files:
    table_name = file.stem
    ehr_tables[table_name] = pd.read_csv(file)

print("Tables loaded:", len(ehr_tables))

In [ ]:
# Display the size of every table
for name, df in ehr_tables.items():
    print(f"{name:25} Rows: {df.shape[0]:8} | Columns: {df.shape[1]:3}")

In [ ]:
ehr_tables["patients"].head()

In [ ]:
ehr_tables["observations"].head()

In [ ]:
ehr_tables["encounters"].head()

In [ ]:
for name in ["patients", "observations", "encounters"]:
    df = ehr_tables[name]

    print(f"\n{'=' * 60}")
    print(f"TABLE: {name}")
    print("=" * 60)

    print("Columns:")
    print(df.columns.tolist())

    print("\nFirst 3 records:")
    display(df.head(3))

    print("\nData types:")
    print(df.dtypes)

In [ ]:
for name, df in ehr_tables.items():
    print(f"{name:25} {df.shape}")
    

In [ ]:
for name in ["patients", "observations", "encounters"]:
    df = ehr_tables[name]

    print(f"\n{'=' * 50}")
    print(f"TABLE: {name}")
    print("=" * 50)

    missing = df.isna().sum()

    report = pd.DataFrame({
        "Missing_Count": missing,
        "Missing_Percentage": (
            missing / len(df) * 100 if len(df) else 0
        )
    })

    display(report[report["Missing_Count"] > 0].sort_values(
        "Missing_Percentage", ascending=False
    ))

In [ ]:
for name in ["patients", "observations", "encounters"]:
    df = ehr_tables[name]

    print(f"\n{name}")
    print("Exact duplicate rows:", df.duplicated().sum())

In [ ]:
print("Patients")
print("Total rows:", len(ehr_tables["patients"]))
print("Unique patient IDs:", ehr_tables["patients"]["Id"].nunique())

print("\nEncounters")
print("Total rows:", len(ehr_tables["encounters"]))
print("Unique encounter IDs:", ehr_tables["encounters"]["Id"].nunique())

In [ ]:
patient_ids = set(ehr_tables["patients"]["Id"].dropna())

observation_patient_ids = set(
    ehr_tables["observations"]["PATIENT"].dropna()
)

encounter_patient_ids = set(
    ehr_tables["encounters"]["PATIENT"].dropna()
)

print(
    "Observation patient IDs missing from patients table:",
    len(observation_patient_ids - patient_ids)
)

print(
    "Encounter patient IDs missing from patients table:",
    len(encounter_patient_ids - patient_ids)
)

In [ ]:
obs = ehr_tables["observations"]

glucose_obs = obs[
    obs["DESCRIPTION"].str.contains(
        "glucose",
        case=False,
        na=False
    )
].copy()

print("Glucose-related records:", len(glucose_obs))

display(
    glucose_obs[
        [
            "DATE",
            "PATIENT",
            "ENCOUNTER",
            "DESCRIPTION",
            "VALUE",
            "UNITS"
        ]
    ].head(20)
)

In [ ]:
display(
    glucose_obs["DESCRIPTION"]
    .value_counts()
    .rename_axis("DESCRIPTION")
    .reset_index(name="RECORD_COUNT")
)

In [ ]:
blood_glucose = glucose_obs[
    glucose_obs["DESCRIPTION"].str.contains(
        "in Blood",
        case=False,
        na=False
    )
].copy()

print("Blood glucose records:", len(blood_glucose))
print("\nMeasurement units:")
display(blood_glucose["UNITS"].value_counts(dropna=False))

print("\nSample records:")
display(
    blood_glucose[
        ["DATE", "PATIENT", "DESCRIPTION", "VALUE", "UNITS"]
    ].head(10)
)

In [ ]:
blood_glucose["PATIENT"].value_counts().describe()

In [ ]:
blood_glucose["DATE"] = pd.to_datetime(
    blood_glucose["DATE"],
    utc=True,
    errors="coerce"
)

patient_timeline = (
    blood_glucose.groupby("PATIENT")["DATE"]
    .agg(["min", "max", "count"])
)

patient_timeline["span_days"] = (
    patient_timeline["max"] - patient_timeline["min"]
).dt.total_seconds() / 86400

display(
    patient_timeline.sort_values("count", ascending=False).head(10)
)

In [ ]:
blood_glucose = blood_glucose.sort_values(
    ["PATIENT", "DATE"]
).copy()

blood_glucose["GAP_HOURS"] = (
    blood_glucose.groupby("PATIENT")["DATE"]
    .diff()
    .dt.total_seconds()
    / 3600
)

display(
    blood_glucose["GAP_HOURS"].describe(
        percentiles=[0.25, 0.50, 0.75, 0.90]
    )
)

In [ ]:
blood_glucose["VALUE_NUMERIC"] = pd.to_numeric(
    blood_glucose["VALUE"],
    errors="coerce"
)

print("Total records:", len(blood_glucose))
print("Missing or non-numeric values:", blood_glucose["VALUE_NUMERIC"].isna().sum())

print("\nGlucose value statistics (mg/dL):")
display(blood_glucose["VALUE_NUMERIC"].describe())

In [ ]:
duplicate_times = blood_glucose[
    blood_glucose.duplicated(
        subset=["PATIENT", "DATE"],
        keep=False
    )
].sort_values(["PATIENT", "DATE"])

print("Records sharing patient and timestamp:", len(duplicate_times))

display(
    duplicate_times[
        ["PATIENT", "DATE", "DESCRIPTION", "VALUE", "UNITS"]
    ].head(20)
)

In [ ]:
print(
    "Exact duplicate rows in blood_glucose:",
    blood_glucose.duplicated().sum()
)

In [ ]:
display(
    duplicate_times[
        duplicate_times["PATIENT"].duplicated(keep=False)
    ].sort_values(["PATIENT", "DATE"]).T
)

In [ ]:
original_columns = ehr_tables["observations"].columns

print(
    "Duplicate patient-time glucose records:",
    blood_glucose.duplicated(
        subset=["PATIENT", "DATE", "CODE", "VALUE", "UNITS"]
    ).sum()
)

In [ ]:
obs = ehr_tables["observations"]

print(
    "Exact duplicate rows in original observations:",
    obs.duplicated().sum()
)

In [ ]:
obs = ehr_tables["observations"]

exact_duplicates = obs[
    obs.duplicated(keep=False)
].sort_values(
    by=list(obs.columns)
)

print("Rows involved in exact duplicates:", len(exact_duplicates))

display(exact_duplicates.head(20))

In [ ]:
# Create a separate working copy; preserve the original data
observations_clean = ehr_tables["observations"].copy()

# Remove exact duplicate rows from the working copy
observations_clean = observations_clean.drop_duplicates().copy()

print("Original rows:", len(ehr_tables["observations"]))
print("Rows after deduplication:", len(observations_clean))
print("Duplicates remaining:", observations_clean.duplicated().sum())
print("Rows removed:", len(ehr_tables["observations"]) - len(observations_clean))

In [ ]:
observations_clean.shape

#Now we will do the data EDA


In [ ]:
observations_clean.shape

In [ ]:
observations_clean.isnull().sum()

In [ ]:
print(
    "Both ENCOUNTER and CATEGORY missing:",
    (
        observations_clean["ENCOUNTER"].isna()
        & observations_clean["CATEGORY"].isna()
    ).sum()
)

In [ ]:
display(
    observations_clean[
        observations_clean["ENCOUNTER"].isna()
    ].head(10)
)

In [ ]:
observations_clean.loc[
    observations_clean["ENCOUNTER"].isna(),
    "DESCRIPTION"
].value_counts()

In [ ]:
encounter_ids = set(ehr_tables["encounters"]["Id"].dropna())
observation_encounter_ids = set(ehr_tables["observations"]["ENCOUNTER"].dropna())

print(
    "Observation encounter IDs not found in encounters:",
    len(observation_encounter_ids - encounter_ids)
)

In [ ]:
print("Total encounters:", len(ehr_tables["encounters"]))
print("Unique encounter IDs:", ehr_tables["encounters"]["Id"].nunique())
print("Duplicate encounter IDs:", ehr_tables["encounters"]["Id"].duplicated().sum())

In [ ]:
ehr_tables["encounters"].isnull().sum()

In [ ]:
ehr_tables["encounters"]["ENCOUNTERCLASS"].value_counts()

In [ ]:
encounters = ehr_tables["encounters"].copy()

encounters["START"] = pd.to_datetime(
    encounters["START"], errors="coerce"
)

encounters["STOP"] = pd.to_datetime(
    encounters["STOP"], errors="coerce"
)

print("Invalid START dates:", encounters["START"].isna().sum())
print("Invalid STOP dates:", encounters["STOP"].isna().sum())
print(
    "STOP before START:",
    (encounters["STOP"] < encounters["START"]).sum()
)

In [ ]:
encounters["DURATION_HOURS"] = (
    encounters["STOP"] - encounters["START"]
).dt.total_seconds() / 3600

encounters["DURATION_HOURS"].describe()

In [ ]:
encounters[
    ["PATIENT", "ENCOUNTERCLASS", "START", "STOP", "DURATION_HOURS"]
].nlargest(10, "DURATION_HOURS")

In [ ]:
encounters.groupby("ENCOUNTERCLASS")["DURATION_HOURS"].agg(
    ["count", "median", "mean", "max"]
).sort_values("median", ascending=False).round(2)

In [ ]:
encounters.loc[
    (encounters["ENCOUNTERCLASS"] == "wellness")
    & (encounters["DURATION_HOURS"] > 24),
    ["PATIENT", "START", "STOP", "DESCRIPTION", "DURATION_HOURS"]
].sort_values("DURATION_HOURS", ascending=False)

In [ ]:
ehr_tables["encounters"].loc[
    ehr_tables["encounters"]["PATIENT"]
    == "edb5a2e3-a981-d33c-5eac-1ca8adde82d3"
]

In [ ]:
wellness_durations = encounters.loc[
    encounters["ENCOUNTERCLASS"] == "wellness",
    "DURATION_HOURS"
]

print("Wellness encounters:", len(wellness_durations))
print("Over 24 hours:", (wellness_durations > 24).sum())
print("Over 12 hours:", (wellness_durations > 12).sum())

In [ ]:
patients = ehr_tables["patients"].copy()

patients["BIRTHDATE"] = pd.to_datetime(
    patients["BIRTHDATE"], errors="coerce"
)

print("Invalid birth dates:", patients["BIRTHDATE"].isna().sum())
print("Future birth dates:", (patients["BIRTHDATE"] > pd.Timestamp.now()).sum())

In [ ]:
reference_date = pd.Timestamp.now().normalize()

patients["AGE"] = (
    reference_date - patients["BIRTHDATE"]
).dt.days / 365.25

patients["AGE"].describe().round(2)

In [ ]:
patients.isnull().sum().sort_values(ascending=False)

In [ ]:
patients.dtypes

In [ ]:
patients["DEATHDATE"] = pd.to_datetime(
    patients["DEATHDATE"], errors="coerce"
)

print("Missing death dates:", patients["DEATHDATE"].isna().sum())
print("Data type:", patients["DEATHDATE"].dtype)

In [ ]:
print(
    "Death dates before birth dates:",
    (
        patients["DEATHDATE"].notna()
        & (patients["DEATHDATE"] < patients["BIRTHDATE"])
    ).sum()
)

In [ ]:
print(
    patients["AGE"].describe().round(2)
)

In [ ]:
print(patients["GENDER"].value_counts(dropna=False))

In [ ]:
print(patients["RACE"].value_counts(dropna=False))

In [ ]:
print(patients["ETHNICITY"].value_counts(dropna=False))

In [ ]:
print(patients["MARITAL"].value_counts(dropna=False))

In [ ]:
print(patients["MARITAL"].unique())

In [ ]:
print("Duplicate patient IDs:", patients["Id"].duplicated().sum())
print("Duplicate patient rows:", patients.duplicated().sum())

In [ ]:
important_columns = [
    "BIRTHDATE",
    "GENDER",
    "RACE",
    "ETHNICITY",
    "MARITAL",
    "HEALTHCARE_EXPENSES",
    "HEALTHCARE_COVERAGE",
    "INCOME"
]

print(patients[important_columns].isnull().sum())

In [ ]:
print(
    patients[
        ["HEALTHCARE_EXPENSES", "HEALTHCARE_COVERAGE", "INCOME"]
    ].describe().round(2)
)

In [ ]:
numeric_cols = [
    "HEALTHCARE_EXPENSES",
    "HEALTHCARE_COVERAGE",
    "INCOME"
]

print((patients[numeric_cols] <= 0).sum())

In [ ]:
print(
    patients.loc[
        patients["HEALTHCARE_COVERAGE"] == 0,
        ["Id", "HEALTHCARE_EXPENSES", "HEALTHCARE_COVERAGE", "INCOME"]
    ]
)

In [ ]:
Q1 = patients["HEALTHCARE_EXPENSES"].quantile(0.25)
Q3 = patients["HEALTHCARE_EXPENSES"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = patients[
    (patients["HEALTHCARE_EXPENSES"] < lower_bound)
    | (patients["HEALTHCARE_EXPENSES"] > upper_bound)
]

print("Lower bound:", round(lower_bound, 2))
print("Upper bound:", round(upper_bound, 2))
print("Potential outliers:", len(outliers))
print(outliers[["HEALTHCARE_EXPENSES"]])

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.boxplot(patients["HEALTHCARE_EXPENSES"], vert=False)
plt.xlabel("Healthcare Expenses")
plt.title("Healthcare Expenses — Outlier Inspection")
plt.grid(axis="x", alpha=0.3)
plt.show()

In [ ]:
Q1 = patients["HEALTHCARE_COVERAGE"].quantile(0.25)
Q3 = patients["HEALTHCARE_COVERAGE"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

coverage_outliers = patients[
    (patients["HEALTHCARE_COVERAGE"] < lower_bound)
    | (patients["HEALTHCARE_COVERAGE"] > upper_bound)
]

print("Lower bound:", round(lower_bound, 2))
print("Upper bound:", round(upper_bound, 2))
print("Potential outliers:", len(coverage_outliers))
print(coverage_outliers[["HEALTHCARE_COVERAGE"]])

In [ ]:
plt.figure(figsize=(8, 4))
plt.boxplot(patients["HEALTHCARE_COVERAGE"], vert=False)
plt.xlabel("Healthcare Coverage")
plt.title("Healthcare Coverage — Outlier Inspection")
plt.grid(axis="x", alpha=0.3)
plt.show()

In [ ]:
Q1 = patients["INCOME"].quantile(0.25)
Q3 = patients["INCOME"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

income_outliers = patients[
    (patients["INCOME"] < lower_bound)
    | (patients["INCOME"] > upper_bound)
]

print("Lower bound:", round(lower_bound, 2))
print("Upper bound:", round(upper_bound, 2))
print("Potential outliers:", len(income_outliers))
print(income_outliers[["INCOME"]])

In [ ]:
plt.figure(figsize=(8, 4))
plt.boxplot(patients["INCOME"], vert=False)
plt.xlabel("Income")
plt.title("Income — Outlier Inspection")
plt.grid(axis="x", alpha=0.3)
plt.show()

In [ ]:
print(
    "Negative healthcare expenses:",
    (patients["HEALTHCARE_EXPENSES"] < 0).sum()
)

print(
    "Negative healthcare coverage:",
    (patients["HEALTHCARE_COVERAGE"] < 0).sum()
)

print(
    "Negative income:",
    (patients["INCOME"] < 0).sum()
)

In [ ]:
conditions = ehr_tables["conditions"].copy()

print("Shape:", conditions.shape)
print("\nColumns:")
print(conditions.columns.tolist())
print("\nFirst 5 rows:")
display(conditions.head())

In [ ]:
print(conditions.isnull().sum())

In [ ]:
print("Duplicate complete rows:", conditions.duplicated().sum())
print("Duplicate condition codes:", conditions["CODE"].duplicated().sum())

In [ ]:
print(
    "Unmatched patient IDs:",
    (~conditions["PATIENT"].isin(patients["Id"])).sum()
)

In [ ]:
print(conditions["DESCRIPTION"].value_counts().head(15))

In [ ]:
prediabetes = conditions[
    conditions["DESCRIPTION"].str.contains(
        "prediabetes", case=False, na=False
    )
]

print("Prediabetes records:", len(prediabetes))
print("Unique patients:", prediabetes["PATIENT"].nunique())

display(prediabetes[["START", "STOP", "PATIENT", "CODE", "DESCRIPTION"]].head())

In [ ]:
print("Prediabetes records:", len(prediabetes))
print("Unique patients:", prediabetes["PATIENT"].nunique())

In [ ]:
diabetes = conditions[
    conditions["DESCRIPTION"].str.contains(
        r"\bdiabetes\b", case=False, na=False, regex=True
    )
]

print("Diabetes records:", len(diabetes))
print("Unique patients:", diabetes["PATIENT"].nunique())
print("\nCondition descriptions:")
print(diabetes["DESCRIPTION"].value_counts())

In [ ]:
cardio = conditions[
    conditions["DESCRIPTION"].str.contains(
        r"cardio|heart disease|hypertension|stroke|coronary",
        case=False,
        na=False,
        regex=True
    )
]

print("Cardiovascular-related records:", len(cardio))
print("Unique patients:", cardio["PATIENT"].nunique())
print("\nCondition descriptions:")
print(cardio["DESCRIPTION"].value_counts().head(15))

In [ ]:
obesity = conditions[
    conditions["DESCRIPTION"].str.contains(
        "obesity", case=False, na=False
    )
]

print("Obesity records:", len(obesity))
print("Unique patients:", obesity["PATIENT"].nunique())
print("\nCondition descriptions:")
print(obesity["DESCRIPTION"].value_counts())

In [ ]:
conditions["START"] = pd.to_datetime(
    conditions["START"],
    errors="coerce"
)

print("Data type:", conditions["START"].dtype)
print("Invalid start dates:", conditions["START"].isna().sum())

In [ ]:
conditions["STOP"] = pd.to_datetime(
    conditions["STOP"],
    errors="coerce"
)

print("Data type:", conditions["STOP"].dtype)
print("Missing STOP dates:", conditions["STOP"].isna().sum())

In [ ]:
invalid_ranges = conditions[
    conditions["STOP"].notna()
    & (conditions["STOP"] < conditions["START"])
]

print("Conditions with STOP before START:", len(invalid_ranges))

In [ ]:
condition_encounters = set(conditions["ENCOUNTER"])
valid_encounters = set(ehr_tables["encounters"]["Id"])

print(
    "Unmatched encounter IDs:",
    len(condition_encounters - valid_encounters)
)

In [ ]:
duplicate_conditions = conditions.duplicated(
    subset=["PATIENT", "CODE", "START"],
    keep=False
)

print("Records involved in potential duplicates:", duplicate_conditions.sum())

In [ ]:
print("Earliest condition start:", conditions["START"].min())
print("Latest condition start:", conditions["START"].max())
print("Unique patients:", conditions["PATIENT"].nunique())
print("Unique condition codes:", conditions["CODE"].nunique())

In [ ]:
conditions["START"].dt.year.value_counts().sort_index().tail(20)

In [ ]:
condition_counts = conditions["PATIENT"].value_counts()

print("Patients with condition records:", condition_counts.size)
print("Minimum records per patient:", condition_counts.min())
print("Median records per patient:", condition_counts.median())
print("Maximum records per patient:", condition_counts.max())

In [ ]:
condition_counts.sort_values(ascending=False).head(10)

In [ ]:
top_patient_id = condition_counts.idxmax()

top_patient_conditions = conditions[
    conditions["PATIENT"] == top_patient_id
]

print("Patient ID:", top_patient_id)
print("\nMost frequent condition descriptions:")
print(top_patient_conditions["DESCRIPTION"].value_counts().head(15))

In [ ]:
medication_reviews = top_patient_conditions[
    top_patient_conditions["DESCRIPTION"]
    == "Medication review due (situation)"
]

print("Total medication-review records:", len(medication_reviews))
print("Unique start dates:", medication_reviews["START"].nunique())
print("\nRecords per start date:")
print(medication_reviews["START"].value_counts().head(10))

In [ ]:
conditions.groupby("DESCRIPTION")["STOP"].apply(
    lambda x: x.isna().sum()
).sort_values(ascending=False).head(10)

In [ ]:
condition_stop_summary = (
    conditions.groupby("DESCRIPTION")
    .agg(
        total_records=("STOP", "size"),
        missing_stop=("STOP", lambda x: x.isna().sum())
    )
)

condition_stop_summary["missing_stop_pct"] = (
    condition_stop_summary["missing_stop"]
    / condition_stop_summary["total_records"]
    * 100
)

condition_stop_summary.sort_values(
    "total_records", ascending=False
).head(10).round(2)

In [ ]:
code_description_counts = (
    conditions.groupby("CODE")["DESCRIPTION"]
    .nunique()
)

print(
    "Codes associated with multiple descriptions:",
    (code_description_counts > 1).sum()
)

In [ ]:
conditions["START_YEAR"] = conditions["START"].dt.year

patient_year_counts = (
    conditions.groupby(["PATIENT", "START_YEAR"])
    .size()
    .reset_index(name="condition_count")
)

print("Patient-year combinations:", len(patient_year_counts))
print("Average conditions per patient-year:", round(
    patient_year_counts["condition_count"].mean(), 2
))
print("Maximum conditions in one patient-year:", patient_year_counts["condition_count"].max())

In [ ]:
patient_year_counts.sort_values(
    "condition_count",
    ascending=False
).head(10)

In [ ]:
dense_patient_id = "5693c080-f485-91e7-54b9-ad9a8c12af62"

dense_patient_2025 = conditions[
    (conditions["PATIENT"] == dense_patient_id)
    & (conditions["START_YEAR"] == 2025)
]

print("Total records:", len(dense_patient_2025))
print("\nCondition descriptions:")
print(dense_patient_2025["DESCRIPTION"].value_counts())

In [ ]:
dense_patient_2025.loc[
    dense_patient_2025["DESCRIPTION"]
    == "Medication review due (situation)",
    ["START", "STOP", "CODE", "ENCOUNTER"]
].sort_values("START")

In [ ]:
encounter_check = dense_patient_2025[
    dense_patient_2025["DESCRIPTION"]
    == "Medication review due (situation)"
].merge(
    ehr_tables["encounters"][["Id", "START", "STOP"]],
    left_on="ENCOUNTER",
    right_on="Id",
    how="left",
    suffixes=("_condition", "_encounter")
)

encounter_check[
    ["START_condition", "STOP_condition",
     "START_encounter", "STOP_encounter"]
].head(10)

In [ ]:
obesity = conditions[
    conditions["DESCRIPTION"].str.contains(
        "obesity", case=False, na=False
    )
]

print("Obesity records:", len(obesity))
print("Unique patients:", obesity["PATIENT"].nunique())
print("\nCondition descriptions:")
print(obesity["DESCRIPTION"].value_counts())

In [ ]:
condition_start_dates = pd.to_datetime(
    encounter_check["START_condition"]
).dt.date

encounter_start_dates = pd.to_datetime(
    encounter_check["START_encounter"],
    utc=True
).dt.date

print(
    "Matching start dates:",
    (condition_start_dates == encounter_start_dates).sum()
)

print(
    "Non-matching start dates:",
    (condition_start_dates != encounter_start_dates).sum()
)

In [ ]:
all_condition_encounters = conditions.merge(
    ehr_tables["encounters"][["Id", "START"]],
    left_on="ENCOUNTER",
    right_on="Id",
    how="left",
    suffixes=("_condition", "_encounter")
)

condition_dates = pd.to_datetime(
    all_condition_encounters["START_condition"]
).dt.date

encounter_dates = pd.to_datetime(
    all_condition_encounters["START_encounter"],
    utc=True
).dt.date

print("Total records checked:", len(all_condition_encounters))
print("Matching start dates:", (condition_dates == encounter_dates).sum())
print("Non-matching start dates:", (condition_dates != encounter_dates).sum())

In [ ]:
date_mismatches = all_condition_encounters.loc[
    condition_dates != encounter_dates,
    [
        "PATIENT",
        "DESCRIPTION",
        "START_condition",
        "START_encounter",
        "ENCOUNTER"
    ]
]

date_mismatches.head(10)

In [ ]:
all_condition_encounters["condition_start_date"] = pd.to_datetime(
    all_condition_encounters["START_condition"]
).dt.normalize()

all_condition_encounters["encounter_start_date"] = pd.to_datetime(
    all_condition_encounters["START_encounter"],
    utc=True
).dt.tz_localize(None).dt.normalize()

all_condition_encounters["start_date_difference_days"] = (
    all_condition_encounters["condition_start_date"]
    - all_condition_encounters["encounter_start_date"]
).dt.days

print(
    all_condition_encounters["start_date_difference_days"]
    .describe()
)

In [ ]:
date_diff_counts = pd.Series({
    "Same date (0 days)": (all_condition_encounters["start_date_difference_days"] == 0).sum(),
    "1–7 days": all_condition_encounters["start_date_difference_days"].between(1, 7).sum(),
    "8–30 days": all_condition_encounters["start_date_difference_days"].between(8, 30).sum(),
    "31–90 days": all_condition_encounters["start_date_difference_days"].between(31, 90).sum(),
    "More than 90 days": (all_condition_encounters["start_date_difference_days"] > 90).sum()
})

display(date_diff_counts)

In [ ]:
large_date_gaps = (
    all_condition_encounters[
        all_condition_encounters["start_date_difference_days"] > 90
    ][
        [
            "PATIENT",
            "DESCRIPTION",
            "START_condition",
            "START_encounter",
            "start_date_difference_days",
            "ENCOUNTER"
        ]
    ]
    .sort_values("start_date_difference_days", ascending=False)
)

display(large_date_gaps)

In [ ]:
large_gaps = all_condition_encounters[
    all_condition_encounters["start_date_difference_days"] > 90
]

print("Records with gaps > 90 days:", len(large_gaps))
print("Unique patients affected:", large_gaps["PATIENT"].nunique())

print("\nRecords per affected patient:")
display(
    large_gaps["PATIENT"]
    .value_counts()
    .rename("large_gap_records")
    .to_frame()
)

In [ ]:
medications = ehr_tables["medications"].copy()

print("Shape:", medications.shape)
print("\nColumns:")
print(medications.columns.tolist())

print("\nFirst 5 rows:")
display(medications.head())

In [ ]:
# ============================================
# MEDICATIONS: INITIAL DATA QUALITY INSPECTION
# ============================================

# 1. Dataset overview
print("=" * 60)
print("1. DATASET OVERVIEW")
print("=" * 60)

print("Shape:", medications.shape)
print("Total rows:", len(medications))
print("Total columns:", len(medications.columns))

print("\nData types:")
display(medications.dtypes.to_frame("Data Type"))


# 2. Missing values
print("\n" + "=" * 60)
print("2. MISSING VALUES")
print("=" * 60)

missing_summary = pd.DataFrame({
    "Missing_Count": medications.isna().sum(),
    "Missing_Percentage": (
        medications.isna().mean() * 100
    ).round(2)
})

display(
    missing_summary.sort_values(
        "Missing_Count",
        ascending=False
    )
)


# 3. Duplicate checks
print("\n" + "=" * 60)
print("3. DUPLICATE CHECKS")
print("=" * 60)

print(
    "Exact duplicate rows:",
    medications.duplicated().sum()
)

print(
    "Duplicate medication IDs:",
    "No dedicated medication ID column exists"
)

# A combination of these columns may identify repeated
# patient-medication records, but repetition is not
# automatically a duplicate.
medication_key = [
    "PATIENT",
    "ENCOUNTER",
    "CODE",
    "START"
]

print(
    "Duplicate patient-encounter-medication-start combinations:",
    medications.duplicated(
        subset=medication_key
    ).sum()
)


# 4. Date validation
print("\n" + "=" * 60)
print("4. DATE VALIDATION")
print("=" * 60)

medications["START"] = pd.to_datetime(
    medications["START"],
    errors="coerce",
    utc=True
)

medications["STOP"] = pd.to_datetime(
    medications["STOP"],
    errors="coerce",
    utc=True
)

print("Invalid or missing START:", medications["START"].isna().sum())
print("Missing STOP:", medications["STOP"].isna().sum())

invalid_dates = (
    medications["START"].notna()
    & medications["STOP"].notna()
    & (medications["STOP"] < medications["START"])
)

print("STOP before START:", invalid_dates.sum())

medications["DURATION_DAYS"] = (
    medications["STOP"] - medications["START"]
).dt.total_seconds() / 86400

print("\nMedication duration in days:")
display(medications["DURATION_DAYS"].describe())


# 5. Numeric validation
print("\n" + "=" * 60)
print("5. NUMERIC COLUMNS")
print("=" * 60)

numeric_columns = [
    "BASE_COST",
    "PAYER_COVERAGE",
    "DISPENSES",
    "TOTALCOST"
]

for col in numeric_columns:
    medications[col] = pd.to_numeric(
        medications[col],
        errors="coerce"
    )

display(medications[numeric_columns].describe().T)

print("\nMissing values after numeric conversion:")
display(medications[numeric_columns].isna().sum().to_frame("Missing_Count"))

print("\nNegative numeric values:")
display(
    (medications[numeric_columns] < 0)
    .sum()
    .to_frame("Negative_Count")
)


# 6. Patient, encounter and payer consistency
print("\n" + "=" * 60)
print("6. REFERENCE ID VALIDATION")
print("=" * 60)

patients_ids = set(ehr_tables["patients"]["Id"].dropna())
encounter_ids = set(ehr_tables["encounters"]["Id"].dropna())
payer_ids = set(ehr_tables["payers"]["Id"].dropna())

print(
    "Unmatched patient IDs:",
    (~medications["PATIENT"].isin(patients_ids)).sum()
)

print(
    "Unmatched encounter IDs:",
    (~medications["ENCOUNTER"].isin(encounter_ids)).sum()
)

print(
    "Unmatched payer IDs:",
    (~medications["PAYER"].isin(payer_ids)).sum()
)


# 7. Most common medications
print("\n" + "=" * 60)
print("7. TOP 15 MEDICATIONS")
print("=" * 60)

display(
    medications["DESCRIPTION"]
    .value_counts(dropna=False)
    .head(15)
    .rename("Record_Count")
    .to_frame()
)


# 8. Cost consistency
print("\n" + "=" * 60)
print("8. COST CONSISTENCY")
print("=" * 60)

cost_comparison = (
    medications["BASE_COST"]
    - medications["TOTALCOST"]
).abs()

print(
    "Rows where BASE_COST differs from TOTALCOST:",
    (cost_comparison > 0.01).sum()
)

print(
    "Rows where payer coverage exceeds total cost:",
    (
        medications["PAYER_COVERAGE"]
        > medications["TOTALCOST"]
    ).sum()
)

print("\nInitial inspection completed.")

In [ ]:
# ============================================
# MEDICATIONS: INVESTIGATE POTENTIAL ANOMALIES
# ============================================

# 1. Ten longest medication durations
print("=" * 70)
print("1. TEN LONGEST MEDICATION DURATIONS")
print("=" * 70)

duration_columns = [
    "PATIENT",
    "DESCRIPTION",
    "START",
    "STOP",
    "DURATION_DAYS",
    "DISPENSES",
    "BASE_COST",
    "TOTALCOST"
]

display(
    medications.nlargest(10, "DURATION_DAYS")[
        duration_columns
    ]
)


# 2. Investigate duplicate medication combinations
print("\n" + "=" * 70)
print("2. POTENTIAL DUPLICATE MEDICATION RECORDS")
print("=" * 70)

duplicate_mask = medications.duplicated(
    subset=medication_key,
    keep=False
)

duplicate_records = medications.loc[
    duplicate_mask
].sort_values(medication_key)

print("Records involved in duplicate combinations:",
      len(duplicate_records))

display(
    duplicate_records[
        [
            "PATIENT",
            "ENCOUNTER",
            "CODE",
            "DESCRIPTION",
            "START",
            "STOP",
            "BASE_COST",
            "PAYER_COVERAGE",
            "DISPENSES",
            "TOTALCOST"
        ]
    ]
)


# 3. Ten highest total costs
print("\n" + "=" * 70)
print("3. TEN HIGHEST TOTAL COST RECORDS")
print("=" * 70)

display(
    medications.nlargest(10, "TOTALCOST")[
        [
            "PATIENT",
            "DESCRIPTION",
            "START",
            "STOP",
            "DISPENSES",
            "BASE_COST",
            "PAYER_COVERAGE",
            "TOTALCOST"
        ]
    ]
)


# 4. Check whether TOTALCOST approximately equals
# BASE_COST multiplied by DISPENSES
print("\n" + "=" * 70)
print("4. COST FORMULA INVESTIGATION")
print("=" * 70)

medications["EXPECTED_TOTAL_FROM_BASE"] = (
    medications["BASE_COST"] * medications["DISPENSES"]
)

medications["COST_DIFFERENCE_FROM_EXPECTED"] = (
    medications["TOTALCOST"]
    - medications["EXPECTED_TOTAL_FROM_BASE"]
)

print(
    "Rows where TOTALCOST differs from "
    "BASE_COST × DISPENSES by more than 0.01:",
    (
        medications["COST_DIFFERENCE_FROM_EXPECTED"].abs()
        > 0.01
    ).sum()
)

print("\nCost formula comparison:")
display(
    medications[
        [
            "BASE_COST",
            "DISPENSES",
            "TOTALCOST",
            "EXPECTED_TOTAL_FROM_BASE",
            "COST_DIFFERENCE_FROM_EXPECTED"
        ]
    ].head(10)
)

In [ ]:
# ============================================
# MEDICATIONS: DURATION AND DISPENSE ANALYSIS
# ============================================

# 1. Duration distribution
print("=" * 60)
print("1. MEDICATION DURATION DISTRIBUTION")
print("=" * 60)

duration_thresholds = [365, 1825, 3650]

for days in duration_thresholds:
    count = (medications["DURATION_DAYS"] > days).sum()
    percentage = count / len(medications) * 100

    print(
        f"Duration > {days} days: "
        f"{count} records ({percentage:.2f}%)"
    )


# 2. Dispense-count distribution
print("\n" + "=" * 60)
print("2. DISPENSE COUNT DISTRIBUTION")
print("=" * 60)

print("\nDispense count statistics:")
display(medications["DISPENSES"].describe())

dispense_thresholds = [10, 50, 100, 500]

for threshold in dispense_thresholds:
    count = (medications["DISPENSES"] > threshold).sum()
    percentage = count / len(medications) * 100

    print(
        f"DISPENSES > {threshold}: "
        f"{count} records ({percentage:.2f}%)"
    )


# 3. Missing STOP by medication
print("\n" + "=" * 60)
print("3. MISSING STOP BY MEDICATION")
print("=" * 60)

missing_stop_summary = (
    medications.assign(STOP_MISSING=medications["STOP"].isna())
    .groupby("DESCRIPTION")
    .agg(
        Total_Records=("DESCRIPTION", "size"),
        Missing_STOP=("STOP_MISSING", "sum")
    )
)

missing_stop_summary["Missing_STOP_Percentage"] = (
    missing_stop_summary["Missing_STOP"]
    / missing_stop_summary["Total_Records"]
    * 100
).round(2)

display(
    missing_stop_summary
    .sort_values(
        ["Missing_STOP", "Total_Records"],
        ascending=False
    )
    .head(15)
)


# 4. Flag candidates for review without deleting anything
print("\n" + "=" * 60)
print("4. RECORDS FLAGGED FOR REVIEW")
print("=" * 60)

medications["REVIEW_LONG_DURATION"] = (
    medications["DURATION_DAYS"] > 3650
)

medications["REVIEW_HIGH_DISPENSES"] = (
    medications["DISPENSES"] > 100
)

print(
    "Duration > 10 years:",
    medications["REVIEW_LONG_DURATION"].sum()
)

print(
    "Dispenses > 100:",
    medications["REVIEW_HIGH_DISPENSES"].sum()
)

print("\nReview flags created. Original medication records retained.")

In [ ]:
# ============================================
# INVESTIGATE HIGH DISPENSE COUNTS
# ============================================

high_dispense_records = medications[
    medications["DISPENSES"] > 100
].copy()

print("Records with DISPENSES > 100:",
      len(high_dispense_records))

print("\nTop medications among high-dispense records:")

display(
    high_dispense_records["DESCRIPTION"]
    .value_counts()
    .head(15)
    .rename("Record_Count")
    .to_frame()
)

print("\nDispense count statistics for these records:")

display(
    high_dispense_records[
        ["DISPENSES", "BASE_COST", "TOTALCOST", "DURATION_DAYS"]
    ].describe().T
)

print("\nTen records with the highest dispense counts:")

display(
    high_dispense_records.nlargest(10, "DISPENSES")[
        [
            "PATIENT",
            "DESCRIPTION",
            "START",
            "STOP",
            "DISPENSES",
            "DURATION_DAYS",
            "BASE_COST",
            "TOTALCOST"
        ]
    ]
)

In [ ]:
# ============================================
# MISSING STOP vs DISPENSE COUNTS
# ============================================

medications["STOP_MISSING"] = medications["STOP"].isna()

stop_dispense_summary = (
    medications.groupby("STOP_MISSING")
    .agg(
        Record_Count=("DISPENSES", "size"),
        Mean_Dispenses=("DISPENSES", "mean"),
        Median_Dispenses=("DISPENSES", "median"),
        Max_Dispenses=("DISPENSES", "max"),
        Mean_TotalCost=("TOTALCOST", "mean"),
        Median_TotalCost=("TOTALCOST", "median")
    )
    .round(2)
)

stop_dispense_summary.index = [
    "STOP available",
    "STOP missing"
]

display(stop_dispense_summary)

In [ ]:
procedures = ehr_tables["procedures"].copy()

print("Shape:", procedures.shape)
print("\nColumns:")
print(procedures.columns.tolist())

print("\nData types:")
display(procedures.dtypes.to_frame("Data Type"))

print("\nFirst 5 rows:")
display(procedures.head())

print("\nMissing values:")
display(
    pd.DataFrame({
        "Missing_Count": procedures.isna().sum(),
        "Missing_Percentage": (
            procedures.isna().mean() * 100
        ).round(2)
    }).sort_values("Missing_Count", ascending=False)
)

print("\nExact duplicate rows:", procedures.duplicated().sum())

In [ ]:
# ============================================
# PROCEDURES: COMPREHENSIVE DATA QUALITY CHECK
# ============================================

# 1. Date conversion and validation
print("=" * 65)
print("1. DATE VALIDATION")
print("=" * 65)

procedures["START"] = pd.to_datetime(
    procedures["START"],
    errors="coerce",
    utc=True
)

procedures["STOP"] = pd.to_datetime(
    procedures["STOP"],
    errors="coerce",
    utc=True
)

print("Invalid or missing START:", procedures["START"].isna().sum())
print("Invalid or missing STOP:", procedures["STOP"].isna().sum())

invalid_duration = (
    procedures["START"].notna()
    & procedures["STOP"].notna()
    & (procedures["STOP"] < procedures["START"])
)

print("STOP before START:", invalid_duration.sum())

procedures["DURATION_MINUTES"] = (
    procedures["STOP"] - procedures["START"]
).dt.total_seconds() / 60

print("\nProcedure duration in minutes:")
display(procedures["DURATION_MINUTES"].describe())


# 2. Potential duplicate records
print("\n" + "=" * 65)
print("2. DUPLICATE CHECKS")
print("=" * 65)

procedure_key = [
    "PATIENT",
    "ENCOUNTER",
    "CODE",
    "START"
]

print("Exact duplicate rows:", procedures.duplicated().sum())

print(
    "Repeated patient-encounter-code-start combinations:",
    procedures.duplicated(subset=procedure_key).sum()
)


# 3. Numeric validation
print("\n" + "=" * 65)
print("3. NUMERIC VALIDATION")
print("=" * 65)

procedures["BASE_COST"] = pd.to_numeric(
    procedures["BASE_COST"],
    errors="coerce"
)

display(procedures["BASE_COST"].describe().to_frame())

print("Missing BASE_COST after conversion:",
      procedures["BASE_COST"].isna().sum())

print("Negative BASE_COST values:",
      (procedures["BASE_COST"] < 0).sum())

print("Zero BASE_COST values:",
      (procedures["BASE_COST"] == 0).sum())


# 4. Reference ID validation
print("\n" + "=" * 65)
print("4. REFERENCE ID VALIDATION")
print("=" * 65)

patients_ids = set(ehr_tables["patients"]["Id"].dropna())
encounter_ids = set(ehr_tables["encounters"]["Id"].dropna())

print(
    "Unmatched patient IDs:",
    (~procedures["PATIENT"].isin(patients_ids)).sum()
)

print(
    "Unmatched encounter IDs:",
    (~procedures["ENCOUNTER"].isin(encounter_ids)).sum()
)


# 5. Most frequent procedures
print("\n" + "=" * 65)
print("5. TOP 15 PROCEDURES")
print("=" * 65)

display(
    procedures["DESCRIPTION"]
    .value_counts()
    .head(15)
    .rename("Record_Count")
    .to_frame()
)


# 6. Procedure date range and yearly distribution
print("\n" + "=" * 65)
print("6. DATE RANGE AND YEARLY DISTRIBUTION")
print("=" * 65)

print("Earliest procedure:", procedures["START"].min())
print("Latest procedure:", procedures["START"].max())

procedures["START_YEAR"] = procedures["START"].dt.year

display(
    procedures["START_YEAR"]
    .value_counts()
    .sort_index()
    .rename("Record_Count")
    .to_frame()
)

print("\nProcedures inspection completed.")

In [ ]:
# ============================================
# PROCEDURES: DURATION AND COST INVESTIGATION
# ============================================

# 1. Inspect the 15 longest procedures
print("=" * 70)
print("1. TOP 15 LONGEST PROCEDURES")
print("=" * 70)

duration_cols = [
    "START", "STOP", "DURATION_MINUTES",
    "DESCRIPTION", "PATIENT", "ENCOUNTER", "BASE_COST"
]

display(
    procedures.nlargest(15, "DURATION_MINUTES")[duration_cols]
)


# 2. Count unusually long procedures
print("\n" + "=" * 70)
print("2. LONG-DURATION COUNTS")
print("=" * 70)

duration_thresholds = [60, 240, 1440, 10080]

for threshold in duration_thresholds:
    count = (procedures["DURATION_MINUTES"] > threshold).sum()
    print(f"Duration > {threshold:>5} minutes: {count}")

# Thresholds:
# 60 minutes = 1 hour
# 240 minutes = 4 hours
# 1440 minutes = 1 day
# 10080 minutes = 7 days


# 3. Inspect zero-duration procedures
print("\n" + "=" * 70)
print("3. ZERO-DURATION PROCEDURES")
print("=" * 70)

zero_duration = procedures[
    procedures["DURATION_MINUTES"] == 0
]

print("Zero-duration records:", len(zero_duration))

display(
    zero_duration[
        ["START", "STOP", "DESCRIPTION", "PATIENT", "ENCOUNTER"]
    ].head(10)
)


# 4. Inspect the 15 most expensive procedures
print("\n" + "=" * 70)
print("4. TOP 15 HIGHEST-COST PROCEDURES")
print("=" * 70)

cost_cols = [
    "BASE_COST", "DESCRIPTION", "START",
    "PATIENT", "ENCOUNTER"
]

display(
    procedures.nlargest(15, "BASE_COST")[cost_cols]
)


# 5. IQR-based cost outlier screening
print("\n" + "=" * 70)
print("5. COST OUTLIER SCREENING")
print("=" * 70)

Q1 = procedures["BASE_COST"].quantile(0.25)
Q3 = procedures["BASE_COST"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)

cost_outliers = procedures[
    (procedures["BASE_COST"] < lower_bound)
    | (procedures["BASE_COST"] > upper_bound)
]

print("Potential cost outliers:", len(cost_outliers))
print(
    "Percentage of records:",
    round(len(cost_outliers) / len(procedures) * 100, 2),
    "%"
)

display(
    cost_outliers.nlargest(15, "BASE_COST")[cost_cols]
)


# 6. Check the most frequent procedure descriptions
print("\n" + "=" * 70)
print("6. PROCEDURE DESCRIPTION FREQUENCIES")
print("=" * 70)

description_counts = procedures["DESCRIPTION"].value_counts()

print("Unique procedure descriptions:", procedures["DESCRIPTION"].nunique())

display(
    description_counts.head(20).rename("Record_Count").to_frame()
)

print("\nDuration and cost investigation completed.")

In [ ]:
# ============================================
# PROCEDURES: CREATE DATA-QUALITY FLAGS
# ============================================

# 1. Flag unusually long durations
procedures["REVIEW_DURATION_OVER_4H"] = (
    procedures["DURATION_MINUTES"] > 240
)

procedures["REVIEW_DURATION_OVER_1D"] = (
    procedures["DURATION_MINUTES"] > 1440
)

procedures["REVIEW_DURATION_OVER_7D"] = (
    procedures["DURATION_MINUTES"] > 10080
)

# 2. Flag zero-duration records
procedures["REVIEW_ZERO_DURATION"] = (
    procedures["DURATION_MINUTES"] == 0
)

# 3. Flag high costs for investigation.
# This is a review flag, NOT proof of an error.
procedures["REVIEW_HIGH_COST"] = (
    procedures["BASE_COST"] > 5000
)

# 4. Flag missing procedure reasons
procedures["REVIEW_MISSING_REASON"] = (
    procedures["REASONCODE"].isna()
    | procedures["REASONDESCRIPTION"].isna()
)

# 5. Summarize flags
flag_columns = [
    "REVIEW_DURATION_OVER_4H",
    "REVIEW_DURATION_OVER_1D",
    "REVIEW_DURATION_OVER_7D",
    "REVIEW_ZERO_DURATION",
    "REVIEW_HIGH_COST",
    "REVIEW_MISSING_REASON",
]

print("PROCEDURE DATA-QUALITY FLAG SUMMARY")
print("=" * 55)

for col in flag_columns:
    count = procedures[col].sum()

    print(
        f"{col}: {count:,} records "
        f"({count / len(procedures) * 100:.2f}%)"
    )

print("\nUpdated procedures shape:", procedures.shape)

In [ ]:
# ============================================
# CLAIMS: INITIAL DATA INSPECTION
# ============================================

claims = ehr_tables["claims"].copy()

# 1. Basic structure
print("=" * 60)
print("1. BASIC INFORMATION")
print("=" * 60)

print("Shape:", claims.shape)
print("\nColumns:")
print(claims.columns.tolist())

print("\nData types:")
display(claims.dtypes.to_frame("Data_Type"))

print("\nFirst 5 rows:")
display(claims.head())


# 2. Missing values
print("\n" + "=" * 60)
print("2. MISSING VALUES")
print("=" * 60)

missing = claims.isna().sum()

missing_summary = pd.DataFrame({
    "Missing_Count": missing,
    "Missing_Percentage": (
        missing / len(claims) * 100
    ).round(2)
})

display(
    missing_summary.sort_values(
        "Missing_Count",
        ascending=False
    )
)


# 3. Duplicate checks
print("\n" + "=" * 60)
print("3. DUPLICATE CHECKS")
print("=" * 60)

print("Exact duplicate rows:", claims.duplicated().sum())

if "Id" in claims.columns:
    print("Unique claim IDs:", claims["Id"].nunique())
    print("Duplicate claim IDs:", claims["Id"].duplicated().sum())


# 4. Numerical summary
print("\n" + "=" * 60)
print("4. NUMERICAL SUMMARY")
print("=" * 60)

numeric_cols = claims.select_dtypes(
    include="number"
).columns.tolist()

if numeric_cols:
    display(claims[numeric_cols].describe().T)
else:
    print("No numerical columns detected.")


# 5. Categorical summary
print("\n" + "=" * 60)
print("5. CATEGORICAL SUMMARY")
print("=" * 60)

categorical_cols = claims.select_dtypes(
    include=["object", "category"]
).columns.tolist()

for col in categorical_cols:
    print(f"\n{col}: {claims[col].nunique()} unique values")
    display(claims[col].value_counts(dropna=False).head(10))


print("\nInitial claims inspection completed.")

In [ ]:
# ============================================
# CLAIMS: DATE, REFERENCE AND LOGICAL VALIDATION
# ============================================

# Keep the original claims DataFrame unchanged.
# Work on a separate copy.
claims_check = claims.copy()

# ------------------------------------------------
# 1. DATE VALIDATION
# ------------------------------------------------

print("=" * 65)
print("1. DATE VALIDATION")
print("=" * 65)

date_cols = [
    "CURRENTILLNESSDATE",
    "SERVICEDATE",
    "LASTBILLEDDATE1",
    "LASTBILLEDDATE2",
    "LASTBILLEDDATEP"
]

for col in date_cols:
    claims_check[col] = pd.to_datetime(
        claims_check[col],
        errors="coerce",
        utc=True
    )

    print(
        f"{col}: invalid dates = "
        f"{claims_check[col].isna().sum()}"
    )

print("\nDate ranges:")

for col in date_cols:
    print(
        f"{col}: "
        f"{claims_check[col].min()} to "
        f"{claims_check[col].max()}"
    )


# ------------------------------------------------
# 2. REFERENCE ID VALIDATION
# ------------------------------------------------

print("\n" + "=" * 65)
print("2. REFERENCE ID VALIDATION")
print("=" * 65)

reference_checks = {
    "PATIENTID": ("patients", "Id"),
    "PROVIDERID": ("providers", "Id"),
    "SUPERVISINGPROVIDERID": ("providers", "Id"),
    "APPOINTMENTID": ("encounters", "Id"),
}

for col, (table_name, id_col) in reference_checks.items():

    valid_ids = set(
        ehr_tables[table_name][id_col].dropna()
    )

    unmatched = (
        claims_check[col].notna()
        & ~claims_check[col].isin(valid_ids)
    ).sum()

    print(f"{col} unmatched references: {unmatched}")


# ------------------------------------------------
# 3. DEPARTMENT REFERENCE CHECKS
# ------------------------------------------------

print("\n" + "=" * 65)
print("3. DEPARTMENT REFERENCE VALIDATION")
print("=" * 65)

department_table = ehr_tables.get("departments")

if department_table is not None:

    print("Department table shape:", department_table.shape)
    print("Department table columns:", department_table.columns.tolist())

    if "Id" in department_table.columns:

        valid_department_ids = set(
            pd.to_numeric(
                department_table["Id"],
                errors="coerce"
            ).dropna()
        )

        for col in [
            "DEPARTMENTID",
            "PATIENTDEPARTMENTID"
        ]:

            values = pd.to_numeric(
                claims_check[col],
                errors="coerce"
            )

            unmatched = (
                values.notna()
                & ~values.isin(valid_department_ids)
            ).sum()

            print(
                f"{col} unmatched references: {unmatched}"
            )

else:
    print("No departments table found in ehr_tables.")


# ------------------------------------------------
# 4. DIAGNOSIS CODE VALIDATION
# ------------------------------------------------

print("\n" + "=" * 65)
print("4. DIAGNOSIS CODE VALIDATION")
print("=" * 65)

diagnosis_cols = [
    f"DIAGNOSIS{i}" for i in range(1, 9)
]

for col in diagnosis_cols:

    values = pd.to_numeric(
        claims_check[col],
        errors="coerce"
    )

    non_missing = values.dropna()

    print(f"\n{col}")
    print("Non-missing values:", len(non_missing))
    print("Unique codes:", non_missing.nunique())
    print("Non-integer numeric values:",
          ((non_missing % 1) != 0).sum())
    print("Zero codes:", (non_missing == 0).sum())
    print("Negative codes:", (non_missing < 0).sum())


# ------------------------------------------------
# 5. CLAIM TYPE AND OUTSTANDING AMOUNTS
# ------------------------------------------------

print("\n" + "=" * 65)
print("5. CLAIM TYPE AND OUTSTANDING AMOUNTS")
print("=" * 65)

claim_type_cols = [
    "HEALTHCARECLAIMTYPEID1",
    "HEALTHCARECLAIMTYPEID2"
]

for col in claim_type_cols:
    print(f"\n{col} value counts:")
    display(
        claims_check[col]
        .value_counts(dropna=False)
        .sort_index()
        .to_frame("Record_Count")
    )


outstanding_cols = [
    "OUTSTANDING1",
    "OUTSTANDING2",
    "OUTSTANDINGP"
]

display(
    claims_check[outstanding_cols]
    .describe()
    .T
)

for col in outstanding_cols:

    values = pd.to_numeric(
        claims_check[col],
        errors="coerce"
    )

    print(
        f"{col}: negative values = "
        f"{(values < 0).sum()}"
    )


# ------------------------------------------------
# 6. STATUS AND SECONDARY INSURANCE RELATIONSHIP
# ------------------------------------------------

print("\n" + "=" * 65)
print("6. STATUS AND INSURANCE RELATIONSHIPS")
print("=" * 65)

print("STATUS1 vs STATUSP:")
display(
    pd.crosstab(
        claims_check["STATUS1"],
        claims_check["STATUSP"],
        dropna=False
    )
)

print("Secondary insurance presence vs STATUS2:")
display(
    pd.crosstab(
        claims_check["SECONDARYPATIENTINSURANCEID"].notna(),
        claims_check["STATUS2"].fillna("MISSING"),
        dropna=False
    )
)

print("\nClaims validation completed.")

In [ ]:
# ============================================================
# CLAIMS: FINAL STRUCTURAL AND LOGICAL CHECKS
# ============================================================

# 1. Compare potentially redundant columns
print("=" * 65)
print("1. COLUMN CONSISTENCY")
print("=" * 65)

print(
    "STATUS1 equals STATUSP:",
    claims["STATUS1"].equals(claims["STATUSP"])
)

print(
    "DEPARTMENTID equals PATIENTDEPARTMENTID:",
    claims["DEPARTMENTID"].equals(claims["PATIENTDEPARTMENTID"])
)

# 2. Inspect diagnosis codes
print("\n" + "=" * 65)
print("2. DIAGNOSIS CODE INSPECTION")
print("=" * 65)

diagnosis_cols = [f"DIAGNOSIS{i}" for i in range(1, 9)]

for col in diagnosis_cols:
    values = claims[col].dropna()

    print(f"\n{col}")
    print("Non-missing records:", len(values))
    print("Unique codes:", values.nunique())
    print("Minimum code:", values.min() if len(values) else None)
    print("Maximum code:", values.max() if len(values) else None)

    # Show unusually large values for manual inspection
    if len(values):
        print("Five largest values:")
        print(values.nlargest(5).to_list())

# 3. Check chronological relationships
print("\n" + "=" * 65)
print("3. CLAIM DATE RELATIONSHIPS")
print("=" * 65)

date_cols = [
    "CURRENTILLNESSDATE",
    "SERVICEDATE",
    "LASTBILLEDDATE1",
    "LASTBILLEDDATE2",
    "LASTBILLEDDATEP"
]

claims_check = claims.copy()

for col in date_cols:
    claims_check[col] = pd.to_datetime(
        claims_check[col],
        errors="coerce",
        utc=True
    )

print(
    "Current illness date after service date:",
    (
        claims_check["CURRENTILLNESSDATE"]
        > claims_check["SERVICEDATE"]
    ).sum()
)

for col in [
    "LASTBILLEDDATE1",
    "LASTBILLEDDATE2",
    "LASTBILLEDDATEP"
]:
    invalid_order = (
        claims_check[col] < claims_check["SERVICEDATE"]
    ).sum()

    print(f"{col} before service date:", invalid_order)

# 4. Check diagnosis codes against conditions
print("\n" + "=" * 65)
print("4. DIAGNOSIS-TO-CONDITIONS CODE OVERLAP")
print("=" * 65)

condition_codes = set(
    conditions["CODE"].astype("string").dropna()
)

for col in diagnosis_cols:
    claim_codes = set(
        claims[col].astype("string").dropna()
    )

    overlap = claim_codes & condition_codes

    print(
        f"{col}:",
        len(overlap),
        "distinct codes also present in conditions"
    )

print("\nFinal structural checks completed.")

In [ ]:
# ============================================================
# CLAIMS: DIAGNOSIS CODE INTEGRITY
# ============================================================

diagnosis_cols = [f"DIAGNOSIS{i}" for i in range(1, 9)]

print("=" * 65)
print("1. ORIGINAL DATA TYPES")
print("=" * 65)

print(claims[diagnosis_cols].dtypes)

print("\n" + "=" * 65)
print("2. LARGEST DIAGNOSIS1 VALUES")
print("=" * 65)

print(
    claims["DIAGNOSIS1"]
    .value_counts(dropna=False)
    .head(10)
)

print("\n" + "=" * 65)
print("3. LARGE DIAGNOSIS2 VALUES")
print("=" * 65)

print(
    claims["DIAGNOSIS2"]
    .dropna()
    .nlargest(10)
    .to_string(index=False)
)

print("\n" + "=" * 65)
print("4. STRING REPRESENTATION")
print("=" * 65)

for col in ["DIAGNOSIS1", "DIAGNOSIS2"]:
    print(f"\n{col} — largest distinct values")

    values = (
        claims[col]
        .dropna()
        .drop_duplicates()
        .nlargest(10)
    )

    for value in values:
        print(
            "Value:", repr(value),
            "| Python type:", type(value).__name__
        )

print("\n" + "=" * 65)
print("5. CONDITION CODE DATA TYPE")
print("=" * 65)

print("Conditions CODE dtype:", conditions["CODE"].dtype)

print("\nSample condition codes:")
print(
    conditions["CODE"]
    .drop_duplicates()
    .head(10)
    .to_list()
)

print("\nDiagnosis-code integrity inspection completed.")


In [ ]:
# ============================================================
# CLAIMS: DIAGNOSIS CODE PRECISION AND OVERLAP
# ============================================================

diagnosis_cols = [f"DIAGNOSIS{i}" for i in range(1, 9)]

print("=" * 60)
print("1. LARGE DIAGNOSIS CODES")
print("=" * 60)

for col in diagnosis_cols:
    values = claims[col].dropna()

    if values.empty:
        print(f"\n{col}: no available values")
        continue

    print(f"\n{col}")
    print("Largest values:")

    for value in values.nlargest(3).unique():
        print(repr(value), type(value).__name__)

print("\n" + "=" * 60)
print("2. DIAGNOSIS CODE OVERLAP")
print("=" * 60)

condition_codes = set(
    conditions["CODE"].dropna().astype("string")
)

for col in diagnosis_cols:
    claim_codes = set(
        claims[col].dropna().astype("string")
    )

    overlap = claim_codes.intersection(condition_codes)

    print(
        f"{col}: {len(overlap)} overlapping distinct codes "
        f"out of {len(claim_codes)} distinct claim codes"
    )

print("\n" + "=" * 60)
print("3. VERIFY FLOAT-TO-INTEGER SAFETY")
print("=" * 60)

for col in diagnosis_cols[1:]:
    values = claims[col].dropna()

    if values.empty:
        print(f"{col}: no values to check")
        continue

    integer_valued = values.mod(1).eq(0).all()
    within_safe_integer_range = (
        values.abs().le(2**53).all()
    )

    print(
        f"{col}: integer-valued={integer_valued}, "
        f"within float exact-integer range="
        f"{within_safe_integer_range}"
    )

print("\nInspection completed.")

In [ ]:
# ============================================================
# VERIFY DIAGNOSIS CODE PRECISION IN ORIGINAL CSV
# ============================================================

from pathlib import Path
import csv

claims_path = DATA_DIR / "claims.csv"

diagnosis_cols = [f"DIAGNOSIS{i}" for i in range(1, 9)]

print("=" * 65)
print("1. ORIGINAL CSV REPRESENTATION")
print("=" * 65)

with claims_path.open("r", encoding="utf-8-sig", newline="") as file:
    reader = csv.DictReader(file)

    for row in reader:
        if (
            row["DIAGNOSIS1"].strip() == "10939881000119105"
            or row["DIAGNOSIS2"].strip() == "10939881000119105"
            or row["DIAGNOSIS3"].strip() == "10939881000119105"
        ):
            print({
                col: row[col]
                for col in diagnosis_cols
                if row[col].strip()
            })

print("\n" + "=" * 65)
print("2. FLOAT PRECISION CHECK")
print("=" * 65)

for col in ["DIAGNOSIS2", "DIAGNOSIS3"]:
    original_values = []

    with claims_path.open(
        "r", encoding="utf-8-sig", newline=""
    ) as file:
        reader = csv.DictReader(file)

        for row in reader:
            value = row[col].strip()

            if value == "10939881000119105":
                original_values.append(value)

    print(f"{col}:")
    print("Exact original code occurrences:", len(original_values))

print("\nOriginal CSV inspection completed.")

In [ ]:
# ============================================================
# CLAIMS: RELOAD WITH EXACT DIAGNOSIS CODE REPRESENTATION
# ============================================================

diagnosis_cols = [f"DIAGNOSIS{i}" for i in range(1, 9)]

claims = pd.read_csv(
    DATA_DIR / "claims.csv",
    dtype={col: "string" for col in diagnosis_cols}
)

print("Claims shape:", claims.shape)

print("\nDiagnosis data types:")
print(claims[diagnosis_cols].dtypes)

print("\nExact large code preserved:")
print(
    claims["DIAGNOSIS3"]
    .eq("10939881000119105")
    .sum()
)

print("\nMissing values in diagnosis columns:")
print(claims[diagnosis_cols].isna().sum())

print("\nExact code verification completed.")



In [ ]:
# ============================================================
# CLAIMS TRANSACTIONS: INITIAL DATA INSPECTION
# ============================================================

transactions = ehr_tables["claims_transactions"].copy()

print("=" * 65)
print("1. DATASET OVERVIEW")
print("=" * 65)

print("Shape:", transactions.shape)
print("\nColumns:")
print(transactions.columns.tolist())

print("\nData types:")
print(transactions.dtypes)

print("\nFirst 5 rows:")
display(transactions.head())

print("\n" + "=" * 65)
print("2. MISSING VALUES")
print("=" * 65)

missing_summary = pd.DataFrame({
    "Missing_Count": transactions.isna().sum(),
    "Missing_Percentage": (
        transactions.isna().mean() * 100
    ).round(2)
})

display(
    missing_summary.sort_values(
        "Missing_Count",
        ascending=False
    )
)

print("\n" + "=" * 65)
print("3. DUPLICATE CHECKS")
print("=" * 65)

print(
    "Exact duplicate rows:",
    transactions.duplicated().sum()
)

for col in transactions.columns:
    if col.lower() == "id":
        print("ID column:", col)
        print("Unique IDs:", transactions[col].nunique())
        print("Missing IDs:", transactions[col].isna().sum())

print("\n" + "=" * 65)
print("4. NUMERICAL SUMMARY")
print("=" * 65)

display(transactions.describe(include="all").T)

print("\nInitial inspection completed.")

In [ ]:
# ============================================================
# CLAIMS TRANSACTIONS: DETAILED VALIDATION
# ============================================================

transactions_check = transactions.copy()

# 1. Transaction type distribution
print("=" * 65)
print("1. TRANSACTION TYPE DISTRIBUTION")
print("=" * 65)

display(
    transactions_check["TYPE"]
    .value_counts(dropna=False)
    .rename_axis("TYPE")
    .to_frame("Record_Count")
)

print("\nFinancial columns by transaction type:")

financial_cols = [
    "AMOUNT",
    "PAYMENTS",
    "ADJUSTMENTS",
    "TRANSFERS",
    "OUTSTANDING"
]

display(
    transactions_check.groupby("TYPE")[financial_cols]
    .agg(["count", "mean", "median", "min", "max"])
    .round(2)
)

# 2. Date validation
print("\n" + "=" * 65)
print("2. DATE VALIDATION")
print("=" * 65)

for col in ["FROMDATE", "TODATE"]:
    parsed = pd.to_datetime(
        transactions_check[col],
        errors="coerce",
        utc=True
    )

    print(f"\n{col}")
    print("Invalid dates:", parsed.isna().sum())
    print("Minimum:", parsed.min())
    print("Maximum:", parsed.max())

    transactions_check[col + "_PARSED"] = parsed

print(
    "\nTODATE before FROMDATE:",
    (
        transactions_check["TODATE_PARSED"]
        < transactions_check["FROMDATE_PARSED"]
    ).sum()
)

# 3. Reference validation
print("\n" + "=" * 65)
print("3. REFERENCE ID VALIDATION")
print("=" * 65)

reference_checks = {
    "CLAIMID": ("claims", "Id"),
    "PATIENTID": ("patients", "Id"),
    "APPOINTMENTID": ("encounters", "Id"),
    "PROVIDERID": ("providers", "Id"),
    "SUPERVISINGPROVIDERID": ("providers", "Id"),
}

for source_col, (table_name, target_col) in reference_checks.items():

    target_table = ehr_tables[table_name]

    valid_ids = set(
        target_table[target_col].dropna().astype(str)
    )

    observed_ids = transactions_check[source_col].dropna().astype(str)

    unmatched = (~observed_ids.isin(valid_ids)).sum()

    print(f"{source_col}: unmatched references = {unmatched}")

# 4. Numeric anomaly screening
print("\n" + "=" * 65)
print("4. NUMERIC ANOMALY SCREENING")
print("=" * 65)

numeric_cols = [
    "AMOUNT",
    "UNITAMOUNT",
    "UNITS",
    "PAYMENTS",
    "ADJUSTMENTS",
    "TRANSFERS",
    "OUTSTANDING"
]

for col in numeric_cols:
    values = pd.to_numeric(
        transactions_check[col],
        errors="coerce"
    )

    print(
        f"{col}: negative={(values < 0).sum()}, "
        f"zero={(values == 0).sum()}, "
        f"missing={values.isna().sum()}, "
        f"maximum={values.max()}"
    )

# 5. Transfer field relationships
print("\n" + "=" * 65)
print("5. TRANSFER FIELD INSPECTION")
print("=" * 65)

print("TRANSFERTYPE values:")
display(
    transactions_check["TRANSFERTYPE"]
    .value_counts(dropna=False)
    .to_frame("Record_Count")
)

print("\nTYPE versus TRANSFERTYPE:")
display(
    pd.crosstab(
        transactions_check["TYPE"],
        transactions_check["TRANSFERTYPE"],
        dropna=False
    )
)

print("\nDetailed transaction validation completed.")

In [ ]:
# ============================================================
# FAST-TRACK: TRANSACTION DATE ANOMALY INVESTIGATION
# ============================================================

# 1. Count date violations by transaction type
print("=" * 60)
print("1. DATE VIOLATIONS BY TRANSACTION TYPE")
print("=" * 60)

transactions_check["DATE_ORDER_ISSUE"] = (
    transactions_check["TODATE_PARSED"]
    < transactions_check["FROMDATE_PARSED"]
)

display(
    transactions_check.groupby("TYPE")[
        "DATE_ORDER_ISSUE"
    ].agg(["sum", "count"])
)

# 2. Inspect date-order issues by transfer type
print("\n" + "=" * 60)
print("2. DATE VIOLATIONS BY TRANSFERTYPE")
print("=" * 60)

display(
    pd.crosstab(
        transactions_check["TYPE"],
        transactions_check["TRANSFERTYPE"],
        values=transactions_check["DATE_ORDER_ISSUE"].astype(int),
        aggfunc="sum",
        dropna=False
    ).fillna(0).astype(int)
)

# 3. Inspect example records
print("\n" + "=" * 60)
print("3. EXAMPLE DATE VIOLATIONS")
print("=" * 60)

display(
    transactions_check.loc[
        transactions_check["DATE_ORDER_ISSUE"],
        [
            "ID", "CLAIMID", "TYPE", "FROMDATE", "TODATE",
            "AMOUNT", "PAYMENTS", "TRANSFERS", "OUTSTANDING"
        ]
    ].head(10)
)

# 4. Check the 1970 placeholder
print("\n" + "=" * 60)
print("4. TODATE EQUAL TO 1970-01-01")
print("=" * 60)

print(
    "Count:",
    transactions_check["TODATE_PARSED"]
    .eq(pd.Timestamp("1970-01-01", tz="UTC"))
    .sum()
)

print("\nInvestigation completed.")

In [ ]:
# ============================================================
# TRANSACTION CLEANING: HANDLE 1970 PLACEHOLDER
# ============================================================

transactions_clean = transactions.copy()

# Parse transaction dates
for col in ["FROMDATE", "TODATE"]:
    transactions_clean[col] = pd.to_datetime(
        transactions_clean[col],
        errors="coerce",
        utc=True
    )

# Count placeholder dates
placeholder = (
    transactions_clean["TODATE"]
    == pd.Timestamp("1970-01-01", tz="UTC")
)

print("Placeholder dates:", placeholder.sum())

# Preserve original dates and replace confirmed placeholder
# dates with missing values in the working copy
transactions_clean["TODATE_ORIGINAL"] = (
    transactions_clean["TODATE"]
)

transactions_clean.loc[placeholder, "TODATE"] = pd.NaT

print(
    "Missing TODATE after placeholder handling:",
    transactions_clean["TODATE"].isna().sum()
)

# Check remaining date-order violations
remaining_issues = (
    transactions_clean["TODATE"].notna()
    & transactions_clean["FROMDATE"].notna()
    & (
        transactions_clean["TODATE"]
        < transactions_clean["FROMDATE"]
    )
)

print("Remaining date-order violations:", remaining_issues.sum())

print("\nCleaned working-copy shape:", transactions_clean.shape)

In [ ]:
# ============================================================
# FINAL TRANSACTION CLEANING SUMMARY
# ============================================================

# Identify columns with only one distinct non-missing value
constant_cols = [
    col for col in transactions_clean.columns
    if transactions_clean[col].nunique(dropna=True) <= 1
]

print("Constant columns:", constant_cols)

# Identify completely empty columns
empty_cols = [
    col for col in transactions_clean.columns
    if transactions_clean[col].isna().all()
]

print("Completely empty columns:", empty_cols)

print("\nFinal shape:", transactions_clean.shape)
print("Exact duplicate rows:", transactions_clean.duplicated().sum())

print("\nTransaction types:")
print(transactions_clean["TYPE"].value_counts())

print("\nCleaning summary completed.")

In [ ]:
# Identify the most promising clinical observations

obs = ehr_tables["observations"].copy()

print("Dataset shape:", obs.shape)

print("\nAvailable columns:")
print(obs.columns.tolist())

print("\nObservation categories:")
print(obs["CATEGORY"].value_counts(dropna=False).head(15))

print("\nMost frequent clinical measurements:")
print(
    obs["DESCRIPTION"]
    .value_counts()
    .head(25)
)

print("\nAvailable measurement units:")
print(
    obs["UNITS"]
    .value_counts(dropna=False)
    .head(15)
)

In [ ]:
# Identify candidate measurements and their patient coverage

obs = ehr_tables["observations"].copy()

keywords = [
    "glucose",
    "blood pressure",
    "heart rate",
    "oxygen saturation",
    "body temperature",
    "respiratory rate",
    "body weight",
    "hemoglobin a1c",
    "hba1c"
]

for keyword in keywords:
    subset = obs[
        obs["DESCRIPTION"]
        .astype("string")
        .str.contains(keyword, case=False, na=False)
    ]

    if not subset.empty:
        print(f"\n{'=' * 55}")
        print("Measurement:", keyword)
        print("Records:", len(subset))
        print("Unique patients:", subset["PATIENT"].nunique())
        print("Unique descriptions:")
        print(subset["DESCRIPTION"].value_counts().head(10))

        print("Available units:")
        print(subset["UNITS"].value_counts(dropna=False).head(5))

In [ ]:
import pandas as pd

obs = ehr_tables["observations"].copy()

obs["DATE"] = pd.to_datetime(
    obs["DATE"],
    errors="coerce",
    utc=True
)

measurements = {
    "Blood glucose": "Glucose [Mass/volume] in Blood",
    "Serum/plasma glucose": "Glucose [Mass/volume] in Serum or Plasma",
    "Systolic BP": "Systolic Blood Pressure",
    "Diastolic BP": "Diastolic Blood Pressure",
    "Heart rate": "Heart rate",
    "Respiratory rate": "Respiratory rate",
    "HbA1c": "Hemoglobin A1c/Hemoglobin.total in Blood",
}

for name, description in measurements.items():
    subset = obs[
        obs["DESCRIPTION"].eq(description)
        & obs["DATE"].notna()
    ].copy()

    patient_stats = (
        subset.groupby("PATIENT")
        .agg(
            records=("DATE", "size"),
            first_date=("DATE", "min"),
            last_date=("DATE", "max"),
            unique_dates=("DATE", "nunique"),
        )
    )

    patient_stats["span_days"] = (
        patient_stats["last_date"]
        - patient_stats["first_date"]
    ).dt.total_seconds() / 86400

    print(f"\n{'=' * 50}")
    print(name)
    print("Records:", len(subset))
    print("Patients:", subset["PATIENT"].nunique())

    if not patient_stats.empty:
        print(
            patient_stats[
                ["records", "unique_dates", "span_days"]
            ].describe().round(2)
        )

In [ ]:
from pathlib import Path
import pandas as pd

# Project directory
PROJECT_DIR = Path(r"C:\Projects_ML\digital-twin")

# Actual preprocessed dataset directory
DATASET_DIR = (
    PROJECT_DIR
    / "data"
    / "wearable"
    / "HUPA-UCM"
    / "HUPA-UCM Diabetes Dataset"
    / "Preprocessed"
)

print("Dataset folder exists:", DATASET_DIR.exists())
print("Dataset path:", DATASET_DIR)

# Find participant CSV files
csv_files = sorted(DATASET_DIR.glob("*.csv"))

print("Number of participant CSV files:", len(csv_files))

# Prevent IndexError if the path is incorrect
if not DATASET_DIR.exists():
    print("\nERROR: Dataset folder not found. Check the path above.")

elif len(csv_files) == 0:
    print("\nERROR: No CSV files found in the Preprocessed folder.")

else:
    print("\nFirst 10 files:")
    for file in csv_files[:10]:
        print(file.name)

    # Inspect the first participant
    df = pd.read_csv(csv_files[0])

    print("\nFirst participant:", csv_files[0].name)
    print("Shape:", df.shape)

    print("\nColumns:")
    print(df.columns.tolist())

    print("\nFirst five rows:")
    display(df.head())

    print("\nData types and missing values:")
    display(
        pd.DataFrame({
            "dtype": df.dtypes.astype(str),
            "missing": df.isna().sum(),
            "missing_pct": (df.isna().mean() * 100).round(2)
        })
    )

In [ ]:
# Read the first participant's CSV correctly

df = pd.read_csv(csv_files[0], sep=";")

print("Participant:", csv_files[0].name)
print("Dataset shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nFirst five rows:")
display(df.head())

print("\nData types and missing values:")
display(
    pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(2)
    })
)

In [ ]:
# Validate all HUPA-UCM preprocessed participant files

import pandas as pd

required_columns = [
    "time",
    "glucose",
    "calories",
    "heart_rate",
    "steps",
    "basal_rate",
    "bolus_volume_delivered",
    "carb_input",
]

validation_results = []

for file in csv_files:
    participant = pd.read_csv(file, sep=";")

    # Parse timestamps
    participant["time"] = pd.to_datetime(
        participant["time"],
        errors="coerce"
    )

    # Calculate time intervals
    intervals = (
        participant["time"]
        .sort_values()
        .diff()
        .dt.total_seconds()
        .div(60)
        .dropna()
    )

    validation_results.append({
        "participant": file.stem,
        "rows": len(participant),
        "missing_cells": int(participant.isna().sum().sum()),
        "missing_columns": [
            col for col in required_columns
            if col not in participant.columns
        ],
        "invalid_timestamps": int(participant["time"].isna().sum()),
        "duplicate_timestamps": int(
            participant["time"].duplicated().sum()
        ),
        "median_interval_min": (
            intervals.median() if not intervals.empty else None
        ),
        "non_5min_intervals": int((intervals != 5).sum()),
        "glucose_min": participant["glucose"].min(),
        "glucose_max": participant["glucose"].max(),
    })

validation_df = pd.DataFrame(validation_results)

display(validation_df)

print("\nTotal participants:", len(validation_df))
print("Total records:", validation_df["rows"].sum())
print("Participants with missing cells:",
      (validation_df["missing_cells"] > 0).sum())
print("Participants with invalid timestamps:",
      (validation_df["invalid_timestamps"] > 0).sum())
print("Participants with duplicate timestamps:",
      (validation_df["duplicate_timestamps"] > 0).sum())

In [ ]:
import pandas as pd

participant_details = []

for file in csv_files:
    temp = pd.read_csv(file, sep=";")

    # Parse timestamps
    temp["time"] = pd.to_datetime(
        temp["time"],
        errors="coerce"
    )

    # Convert glucose to numeric for validation
    glucose = pd.to_numeric(
        temp["glucose"],
        errors="coerce"
    )

    # Calculate observation duration
    first_time = temp["time"].min()
    last_time = temp["time"].max()

    duration_days = (
        (last_time - first_time).total_seconds() / 86400
        if pd.notna(first_time) and pd.notna(last_time)
        else None
    )

    participant_details.append({
        "participant": file.stem,
        "records": len(temp),
        "first_timestamp": first_time,
        "last_timestamp": last_time,
        "duration_days": round(duration_days, 2)
            if duration_days is not None else None,
        "glucose_missing": int(glucose.isna().sum()),
        "glucose_min": glucose.min(),
        "glucose_median": glucose.median(),
        "glucose_max": glucose.max(),
        "below_70": int((glucose < 70).sum()),
        "above_180": int((glucose > 180).sum()),
        "above_250": int((glucose > 250).sum()),
    })

participant_summary = pd.DataFrame(participant_details)

# Display largest datasets first
display(
    participant_summary.sort_values(
        "records",
        ascending=False
    ).reset_index(drop=True)
)

print("\n--- Overall Summary ---")
print("Participants:", len(participant_summary))
print("Total records:", participant_summary["records"].sum())
print(
    "Total observation span (median days):",
    round(participant_summary["duration_days"].median(), 2)
)
print(
    "Participants with glucose conversion failures:",
    (participant_summary["glucose_missing"] > 0).sum()
)

In [ ]:
import pandas as pd

# Settings
HORIZON_STEPS = 24  # 2 hours at 5-minute intervals

feature_columns = [
    "glucose",
    "calories",
    "heart_rate",
    "steps",
    "basal_rate",
    "bolus_volume_delivered",
    "carb_input",
]

participant_frames = []

for file in csv_files:
    temp = pd.read_csv(file, sep=";")

    temp["time"] = pd.to_datetime(
        temp["time"],
        errors="coerce"
    )

    # Sort chronologically
    temp = temp.sort_values("time").reset_index(drop=True)

    # Keep participant identity
    temp["participant"] = file.stem

    # Create future glucose target
    temp["glucose_future_2h"] = (
        temp["glucose"].shift(-HORIZON_STEPS)
    )

    participant_frames.append(temp)

# Combine participants in memory
prediction_data = pd.concat(
    participant_frames,
    ignore_index=True
)

# Remove rows without a future target
prediction_data = prediction_data.dropna(
    subset=["glucose_future_2h"]
).reset_index(drop=True)

print("Prediction dataset shape:", prediction_data.shape)
print("Participants:", prediction_data["participant"].nunique())

print("\nTarget summary:")
display(prediction_data["glucose_future_2h"].describe())

print("\nRecords per participant:")
display(
    prediction_data.groupby("participant")
    .size()
    .sort_values(ascending=False)
    .to_frame("prediction_rows")
)

print("\nFirst five rows:")
display(prediction_data.head())

In [ ]:
# Validate that each target is exactly 2 hours after its input timestamp

HORIZON_MINUTES = 120

horizon_check = prediction_data.copy()

# Reconstruct the timestamp 24 rows ahead within each participant
horizon_check["future_time"] = (
    horizon_check.groupby("participant")["time"]
    .shift(-24)
)

horizon_check["actual_horizon_minutes"] = (
    horizon_check["future_time"] - horizon_check["time"]
).dt.total_seconds() / 60

# Check actual time difference
valid_horizon = (
    horizon_check["actual_horizon_minutes"] == HORIZON_MINUTES
)

print("Prediction rows:", len(horizon_check))
print("Exactly 120 minutes ahead:", int(valid_horizon.sum()))
print("Other time differences:", int((~valid_horizon).sum()))

print("\nActual horizon distribution:")
display(
    horizon_check["actual_horizon_minutes"]
    .value_counts(dropna=False)
    .head(10)
)

In [ ]:
# Rebuild the 2-hour prediction target within each participant

HORIZON_STEPS = 24

participant_frames = []

for file in csv_files:
    temp = pd.read_csv(file, sep=";")

    temp["time"] = pd.to_datetime(temp["time"], errors="coerce")
    temp = temp.sort_values("time").reset_index(drop=True)

    # Target must come from the same participant
    temp["glucose_future_2h"] = temp["glucose"].shift(-HORIZON_STEPS)

    # Store only rows with an available future target
    temp = temp.dropna(subset=["glucose_future_2h"])

    participant_frames.append(temp)

prediction_data = pd.concat(
    participant_frames,
    ignore_index=True
)

print("Prediction dataset shape:", prediction_data.shape)
print("Participants:", prediction_data["participant"].nunique()
      if "participant" in prediction_data.columns else len(csv_files))

# Validate target horizon within each source file
horizon_results = []

for file in csv_files:
    temp = pd.read_csv(file, sep=";")
    temp["time"] = pd.to_datetime(temp["time"], errors="coerce")
    temp = temp.sort_values("time").reset_index(drop=True)

    future_time = temp["time"].shift(-HORIZON_STEPS)
    minutes = (future_time - temp["time"]).dt.total_seconds() / 60

    horizon_results.append({
        "participant": file.stem,
        "exact_120_min_rows": int((minutes == 120).sum()),
        "other_or_missing": int((minutes != 120).sum()),
    })

horizon_summary = pd.DataFrame(horizon_results)
display(horizon_summary)

print("\nTotal exact 120-minute targets:",
      horizon_summary["exact_120_min_rows"].sum())

In [ ]:
# Inspect the dataset we created earlier
print("Dataset shape:", prediction_data.shape)

print("\nColumn names:")
print(prediction_data.columns.tolist())

print("\nFirst 5 rows:")
display(prediction_data.head())

print("\nData types:")
print(prediction_data.dtypes)

In [ ]:
import pandas as pd
import numpy as np

df = prediction_data.copy()

TARGET = "glucose_future_2h"
TIME = "time"

# 1. Dataset overview
print("Dataset shape:", df.shape)
print("Unique timestamps:", df[TIME].nunique())

# 2. Missing values
print("\nMissing values:")
print(df.isna().sum())

# 3. Target distribution
print("\nFuture glucose statistics:")
print(df[TARGET].describe())

# 4. Review suspicious glucose values (do not delete automatically)
print("\nCurrent glucose outside 40–400 mg/dL:",
      ((df["glucose"] < 40) | (df["glucose"] > 400)).sum())

print("Future glucose outside 40–400 mg/dL:",
      ((df[TARGET] < 40) | (df[TARGET] > 400)).sum())

# 5. Feature correlations with target
numeric_cols = df.select_dtypes(include="number").columns

print("\nCorrelation with future glucose:")
display(
    df[numeric_cols]
    .corr()[TARGET]
    .sort_values(ascending=False)
)

# 6. Check feature availability
print("\nFeature columns:")
print([c for c in df.columns if c not in [TIME, TARGET]])

In [ ]:
import pandas as pd

HORIZON_STEPS = 24
participant_frames = []

for file in csv_files:
    temp = pd.read_csv(file, sep=";")

    temp["time"] = pd.to_datetime(temp["time"], errors="coerce")
    temp["glucose"] = pd.to_numeric(temp["glucose"], errors="coerce")

    temp = temp.dropna(subset=["time", "glucose"])
    temp = temp.sort_values("time").reset_index(drop=True)

    # Restore participant identity
    temp["participant"] = file.stem

    # Create future target within this participant's file
    temp["glucose_future_2h"] = temp["glucose"].shift(-HORIZON_STEPS)

    # Calculate actual prediction horizon
    future_time = temp["time"].shift(-HORIZON_STEPS)
    temp["horizon_minutes"] = (
        (future_time - temp["time"]).dt.total_seconds() / 60
    )

    # Keep only genuine 120-minute targets
    temp = temp[temp["horizon_minutes"] == 120].copy()

    participant_frames.append(temp)

prediction_data = pd.concat(participant_frames, ignore_index=True)

print("Validated dataset shape:", prediction_data.shape)
print("Participants:", prediction_data["participant"].nunique())

print("\nRows per participant:")
display(
    prediction_data.groupby("participant")
    .size()
    .sort_values(ascending=False)
)

print("\nMissing values:")
print(prediction_data.isna().sum())

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Prepare data
df = prediction_data.copy()

FEATURES = [
    "glucose",
    "calories",
    "heart_rate",
    "steps",
    "basal_rate",
    "bolus_volume_delivered",
    "carb_input"
]

TARGET = "glucose_future_2h"

X = df[FEATURES]
y = df[TARGET]
groups = df["participant"]

# 2. Split by participant: no participant appears in both sets
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

train_participants = set(groups.iloc[train_idx])
test_participants = set(groups.iloc[test_idx])

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Training participants:", len(train_participants))
print("Testing participants:", len(test_participants))
print("Participant overlap:", train_participants & test_participants)

# 3. Persistence baseline
baseline_predictions = X_test["glucose"].to_numpy()

baseline_mae = mean_absolute_error(y_test, baseline_predictions)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_predictions))

print("\n--- Persistence Baseline ---")
print(f"MAE:  {baseline_mae:.2f}")
print(f"RMSE: {baseline_rmse:.2f}")

# 4. Train Random Forest
model = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,
    min_samples_leaf=5,
    n_jobs=-1,
    random_state=42
)

model.fit(X_train, y_train)

predictions = model.predict(X_test)

# 5. Evaluate
model_mae = mean_absolute_error(y_test, predictions)
model_rmse = np.sqrt(mean_squared_error(y_test, predictions))
model_r2 = r2_score(y_test, predictions)

print("\n--- Random Forest Results ---")
print(f"MAE:  {model_mae:.2f}")
print(f"RMSE: {model_rmse:.2f}")
print(f"R²:   {model_r2:.4f}")

# 6. Feature importance
importance = pd.Series(
    model.feature_importances_,
    index=FEATURES
).sort_values(ascending=False)

print("\n--- Feature Importance ---")
display(importance.to_frame("importance"))

# 7. Compare models
print("\n--- Comparison ---")
print(f"Baseline MAE:     {baseline_mae:.2f}")
print(f"Random Forest MAE: {model_mae:.2f}")
print(f"MAE improvement:   {baseline_mae - model_mae:.2f}")

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error

# 1. Inspect the participant split
print("TRAINING PARTICIPANTS:")
print(sorted(train_participants))

print("\nTESTING PARTICIPANTS:")
print(sorted(test_participants))

print("\nROWS PER PARTICIPANT IN THE FULL DATASET:")
display(
    prediction_data.groupby("participant")
    .size()
    .sort_values(ascending=False)
    .to_frame("rows")
)

# 2. Build a test-results dataframe
test_results = prediction_data.iloc[test_idx][
    ["participant", "glucose", "glucose_future_2h"]
].copy()

test_results["baseline_prediction"] = baseline_predictions
test_results["rf_prediction"] = predictions

# 3. Calculate metrics for each test participant
participant_metrics = []

for participant, group in test_results.groupby("participant"):

    baseline_mae = mean_absolute_error(
        group["glucose_future_2h"],
        group["baseline_prediction"]
    )

    rf_mae = mean_absolute_error(
        group["glucose_future_2h"],
        group["rf_prediction"]
    )

    baseline_rmse = np.sqrt(
        mean_squared_error(
            group["glucose_future_2h"],
            group["baseline_prediction"]
        )
    )

    rf_rmse = np.sqrt(
        mean_squared_error(
            group["glucose_future_2h"],
            group["rf_prediction"]
        )
    )

    participant_metrics.append({
        "participant": participant,
        "test_rows": len(group),
        "baseline_MAE": baseline_mae,
        "RF_MAE": rf_mae,
        "baseline_RMSE": baseline_rmse,
        "RF_RMSE": rf_rmse,
        "MAE_improvement": baseline_mae - rf_mae
    })

participant_metrics = pd.DataFrame(participant_metrics)

print("\nPER-PARTICIPANT MODEL PERFORMANCE:")
display(
    participant_metrics.sort_values("test_rows", ascending=False)
)

print("\nMEAN PARTICIPANT-LEVEL MAE:")
print(
    "Baseline:",
    round(participant_metrics["baseline_MAE"].mean(), 2)
)

print(
    "Random Forest:",
    round(participant_metrics["RF_MAE"].mean(), 2)
)

print("\nPositive MAE improvement means Random Forest performed better.")

In [ ]:
import pandas as pd

# Check the original files, preserving participant boundaries
sampling_results = []

for file in csv_files:
    temp = pd.read_csv(file, sep=";")

    temp["time"] = pd.to_datetime(temp["time"], errors="coerce")
    temp = temp.dropna(subset=["time"]).sort_values("time")

    gaps = temp["time"].diff().dt.total_seconds() / 60

    sampling_results.append({
        "participant": file.stem,
        "rows": len(temp),
        "median_interval_min": gaps.median(),
        "interval_5min_pct": (gaps == 5).mean() * 100,
        "interval_10min_pct": (gaps == 10).mean() * 100,
        "gaps_over_10min": int((gaps > 10).sum()),
        "max_gap_min": gaps.max()
    })

sampling_report = pd.DataFrame(sampling_results)

print("Sampling interval report:")
display(
    sampling_report.sort_values("rows", ascending=False)
)

print("\nOverall dataset summary:")
print("Participants:", len(sampling_report))
print("Total source rows:", sampling_report["rows"].sum())
print(
    "Participants with median interval of 5 minutes:",
    (sampling_report["median_interval_min"] == 5).sum()
)

In [ ]:
import pandas as pd
import numpy as np

# Work with the validated prediction dataset
df = prediction_data.copy()

df["time"] = pd.to_datetime(df["time"])
df = df.sort_values(["participant", "time"]).reset_index(drop=True)

# Create historical features within each participant
grouped_glucose = df.groupby("participant")["glucose"]

# Historical glucose values
for lag in [1, 3, 6, 12]:
    df[f"glucose_lag_{lag}"] = grouped_glucose.shift(lag)

# Rolling features: exclude the current reading
for window in [3, 6]:
    df[f"glucose_mean_{window}"] = (
        df.groupby("participant")["glucose"]
        .transform(
            lambda s: s.shift(1).rolling(window).mean()
        )
    )

df["glucose_std_6"] = (
    df.groupby("participant")["glucose"]
    .transform(
        lambda s: s.shift(1).rolling(6).std()
    )
)

# Glucose trends
df["glucose_change_15m"] = (
    df["glucose"] - df["glucose_lag_3"]
)

df["glucose_change_30m"] = (
    df["glucose"] - df["glucose_lag_6"]
)

# Check results
new_features = [
    "glucose_lag_1",
    "glucose_lag_3",
    "glucose_lag_6",
    "glucose_lag_12",
    "glucose_mean_3",
    "glucose_mean_6",
    "glucose_std_6",
    "glucose_change_15m",
    "glucose_change_30m"
]

print("Dataset shape:", df.shape)
print("Participants:", df["participant"].nunique())

print("\nMissing values in new features:")
print(df[new_features].isna().sum())

print("\nSample of engineered features:")
display(df[["participant", "time", "glucose"] + new_features].head(15))

In [ ]:
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Define original and engineered features
original_features = [
    "glucose",
    "calories",
    "heart_rate",
    "steps",
    "basal_rate",
    "bolus_volume_delivered",
    "carb_input"
]

historical_features = [
    "glucose_lag_1",
    "glucose_lag_3",
    "glucose_lag_6",
    "glucose_lag_12",
    "glucose_mean_3",
    "glucose_mean_6",
    "glucose_std_6",
    "glucose_change_15m",
    "glucose_change_30m"
]

TARGET = "glucose_future_2h"

# 2. Remove rows lacking historical features
model_df = df.dropna(
    subset=historical_features + [TARGET]
).copy()

print("Original rows:", len(df))
print("Rows available for modeling:", len(model_df))
print("Rows removed:", len(df) - len(model_df))

# 3. Preserve the same participant split
train_mask = model_df["participant"].isin(train_participants)
test_mask = model_df["participant"].isin(test_participants)

X_train_v2 = model_df.loc[
    train_mask, original_features + historical_features
]
y_train_v2 = model_df.loc[train_mask, TARGET]

X_test_v2 = model_df.loc[
    test_mask, original_features + historical_features
]
y_test_v2 = model_df.loc[test_mask, TARGET]

print("\nTraining rows:", len(X_train_v2))
print("Testing rows:", len(X_test_v2))

# 4. Persistence baseline
baseline_v2 = X_test_v2["glucose"].to_numpy()

baseline_mae_v2 = mean_absolute_error(y_test_v2, baseline_v2)
baseline_rmse_v2 = np.sqrt(
    mean_squared_error(y_test_v2, baseline_v2)
)

# 5. Train Random Forest V2
model_v2 = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,
    min_samples_leaf=5,
    n_jobs=-1,
    random_state=42
)

model_v2.fit(X_train_v2, y_train_v2)

pred_v2 = model_v2.predict(X_test_v2)

rf_mae_v2 = mean_absolute_error(y_test_v2, pred_v2)
rf_rmse_v2 = np.sqrt(
    mean_squared_error(y_test_v2, pred_v2)
)
rf_r2_v2 = r2_score(y_test_v2, pred_v2)

# 6. Results
results_v2 = pd.DataFrame({
    "Model": ["Persistence Baseline", "Random Forest V2"],
    "MAE": [baseline_mae_v2, rf_mae_v2],
    "RMSE": [baseline_rmse_v2, rf_rmse_v2]
})

print("\nMODEL COMPARISON:")
display(results_v2.round(3))

print(f"Random Forest V2 R²: {rf_r2_v2:.4f}")
print(
    "MAE improvement over baseline:",
    round(baseline_mae_v2 - rf_mae_v2, 3)
)

# 7. Feature importance
importance_v2 = pd.Series(
    model_v2.feature_importances_,
    index=original_features + historical_features
).sort_values(ascending=False)

print("\nTOP 10 FEATURES:")
display(importance_v2.head(10).to_frame("importance"))

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Predictions from the V2 test set
evaluation_df = model_df.loc[
    test_mask,
    ["participant", "glucose", "glucose_future_2h"]
].copy()

evaluation_df["baseline_prediction"] = (
    X_test_v2["glucose"].to_numpy()
)

evaluation_df["rf_v2_prediction"] = pred_v2

# Evaluate each participant separately
participant_results = []

for participant, group in evaluation_df.groupby("participant"):

    actual = group["glucose_future_2h"]

    baseline_pred = group["baseline_prediction"]
    rf_pred = group["rf_v2_prediction"]

    baseline_mae = mean_absolute_error(actual, baseline_pred)
    rf_mae = mean_absolute_error(actual, rf_pred)

    baseline_rmse = np.sqrt(
        mean_squared_error(actual, baseline_pred)
    )

    rf_rmse = np.sqrt(
        mean_squared_error(actual, rf_pred)
    )

    participant_results.append({
        "participant": participant,
        "test_rows": len(group),
        "baseline_MAE": baseline_mae,
        "RF_V2_MAE": rf_mae,
        "baseline_RMSE": baseline_rmse,
        "RF_V2_RMSE": rf_rmse,
        "MAE_improvement": baseline_mae - rf_mae
    })

participant_results = pd.DataFrame(participant_results)

display(
    participant_results
    .sort_values("test_rows", ascending=False)
    .round(3)
)

print("\nMean participant-level MAE:")
print(
    "Baseline:",
    round(participant_results["baseline_MAE"].mean(), 3)
)
print(
    "Random Forest V2:",
    round(participant_results["RF_V2_MAE"].mean(), 3)
)

print("\nParticipants where V2 improved MAE:")
print(
    (participant_results["MAE_improvement"] > 0).sum(),
    "out of",
    len(participant_results)
)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# evaluation_df already contains predictions for the same test rows
actual = evaluation_df["glucose_future_2h"]

baseline_pred = evaluation_df["baseline_prediction"]
rf_pred = evaluation_df["rf_v2_prediction"]

# Overall, row-weighted metrics
overall_results = pd.DataFrame({
    "Metric": ["MAE", "RMSE", "R2"],
    "Persistence Baseline": [
        mean_absolute_error(actual, baseline_pred),
        np.sqrt(mean_squared_error(actual, baseline_pred)),
        r2_score(actual, baseline_pred)
    ],
    "Random Forest V2": [
        mean_absolute_error(actual, rf_pred),
        np.sqrt(mean_squared_error(actual, rf_pred)),
        r2_score(actual, rf_pred)
    ]
})

print("OVERALL ROW-WEIGHTED RESULTS")
display(overall_results.round(3))

# Equal-weight participant-level metrics
print("\nPARTICIPANT-LEVEL SUMMARY")

print(
    "Baseline mean MAE:",
    round(participant_results["baseline_MAE"].mean(), 3)
)

print(
    "Random Forest V2 mean MAE:",
    round(participant_results["RF_V2_MAE"].mean(), 3)
)

print(
    "Participants improved:",
    int((participant_results["MAE_improvement"] > 0).sum()),
    "out of",
    len(participant_results)
)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# evaluation_df already contains predictions for the same test rows
actual = evaluation_df["glucose_future_2h"]

baseline_pred = evaluation_df["baseline_prediction"]
rf_pred = evaluation_df["rf_v2_prediction"]

# Overall, row-weighted metrics
overall_results = pd.DataFrame({
    "Metric": ["MAE", "RMSE", "R2"],
    "Persistence Baseline": [
        mean_absolute_error(actual, baseline_pred),
        np.sqrt(mean_squared_error(actual, baseline_pred)),
        r2_score(actual, baseline_pred)
    ],
    "Random Forest V2": [
        mean_absolute_error(actual, rf_pred),
        np.sqrt(mean_squared_error(actual, rf_pred)),
        r2_score(actual, rf_pred)
    ]
})

print("OVERALL ROW-WEIGHTED RESULTS")
display(overall_results.round(3))

# Equal-weight participant-level metrics
print("\nPARTICIPANT-LEVEL SUMMARY")

print(
    "Baseline mean MAE:",
    round(participant_results["baseline_MAE"].mean(), 3)
)

print(
    "Random Forest V2 mean MAE:",
    round(participant_results["RF_V2_MAE"].mean(), 3)
)

print(
    "Participants improved:",
    int((participant_results["MAE_improvement"] > 0).sum()),
    "out of",
    len(participant_results)
)

In [ ]:
# Check the biggest prediction errors

error_analysis = evaluation_df.copy()

error_analysis["error"] = (
    error_analysis["glucose_future_2h"]
    - error_analysis["rf_v2_prediction"]
)

error_analysis["absolute_error"] = error_analysis["error"].abs()

display(
    error_analysis.nlargest(10, "absolute_error")[
        [
            "participant",
            "glucose",
            "glucose_future_2h",
            "rf_v2_prediction",
            "absolute_error"
        ]
    ].round(2)
)

In [ ]:
# Categorize actual future glucose values

error_analysis["glucose_range"] = pd.cut(
    error_analysis["glucose_future_2h"],
    bins=[-float("inf"), 70, 180, float("inf")],
    labels=["Below 70", "70–180", "Above 180"]
)

# Calculate prediction error for each range

summary = error_analysis.groupby(
    "glucose_range",
    observed=False
).agg(
    total_rows=("absolute_error", "size"),
    mean_absolute_error=("absolute_error", "mean")
).round(2)

display(summary)

In [ ]:
# Investigate large glucose increases

error_analysis["actual_glucose_change"] = (
    error_analysis["glucose_future_2h"]
    - error_analysis["glucose"]
)

error_analysis["predicted_glucose_change"] = (
    error_analysis["rf_v2_prediction"]
    - error_analysis["glucose"]
)

print("Actual glucose change over 2 hours:")
display(
    error_analysis["actual_glucose_change"].describe().round(2)
)

print("\nLargest actual glucose increases:")
display(
    error_analysis.nlargest(10, "actual_glucose_change")[
        [
            "participant",
            "glucose",
            "glucose_future_2h",
            "actual_glucose_change",
            "rf_v2_prediction",
            "absolute_error"
        ]
    ].round(2)
)

In [ ]:
# Examine historical trends before large glucose increases

error_analysis["glucose_change_15m"] = (
    model_df.loc[
        evaluation_df.index, "glucose_change_15m"
    ].to_numpy()
)

error_analysis["glucose_change_30m"] = (
    model_df.loc[
        evaluation_df.index, "glucose_change_30m"
    ].to_numpy()
)

# Identify large increases over the next two hours
large_increases = error_analysis[
    error_analysis["actual_glucose_change"] >= 100
].copy()

print("Number of large increases:", len(large_increases))

print("\nCurrent glucose before large increases:")
print(large_increases["glucose"].describe().round(2))

print("\nGlucose change during previous 15 minutes:")
print(large_increases["glucose_change_15m"].describe().round(2))

print("\nGlucose change during previous 30 minutes:")
print(large_increases["glucose_change_30m"].describe().round(2))

print("\nCompare with all test records:")

print(
    "All-record mean 15-minute change:",
    round(error_analysis["glucose_change_15m"].mean(), 2)
)

print(
    "Large-increase mean 15-minute change:",
    round(large_increases["glucose_change_15m"].mean(), 2)
)

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report

# Actual and predicted glucose changes
actual_change = (
    error_analysis["glucose_future_2h"] - error_analysis["glucose"]
)

predicted_change = (
    error_analysis["rf_v2_prediction"] - error_analysis["glucose"]
)

# Define large increases
actual_event = actual_change >= 100
predicted_event = predicted_change >= 100

# Compare event detection
tn, fp, fn, tp = confusion_matrix(
    actual_event,
    predicted_event,
    labels=[False, True]
).ravel()

print("Actual large increases:", int(actual_event.sum()))
print("Predicted large increases:", int(predicted_event.sum()))

print("\nConfusion matrix:")
print("True negatives:", tn)
print("False positives:", fp)
print("False negatives:", fn)
print("True positives:", tp)

if tp + fn > 0:
    print("\nEvent recall:", round(tp / (tp + fn), 4))

if tp + fp > 0:
    print("Event precision:", round(tp / (tp + fp), 4))

In [ ]:
import pandas as pd
import numpy as np

error_analysis["actual_change"] = (
    error_analysis["glucose_future_2h"]
    - error_analysis["glucose"]
)

error_analysis["predicted_change"] = (
    error_analysis["rf_v2_prediction"]
    - error_analysis["glucose"]
)

# Summarize actual vs predicted changes
print("ACTUAL GLUCOSE CHANGE")
print(error_analysis["actual_change"].describe().round(2))

print("\nPREDICTED GLUCOSE CHANGE")
print(error_analysis["predicted_change"].describe().round(2))

# Compare predictions for large actual increases
large_increases = error_analysis[
    error_analysis["actual_change"] >= 100
]

print("\nLARGE-INCREASE ANALYSIS")
print("Number of events:", len(large_increases))
print(
    "Mean actual increase:",
    round(large_increases["actual_change"].mean(), 2)
)
print(
    "Mean predicted increase:",
    round(large_increases["predicted_change"].mean(), 2)
)
print(
    "Median predicted increase:",
    round(large_increases["predicted_change"].median(), 2)
)

In [ ]:
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    average_precision_score,
    roc_auc_score
)

# 1. Define the experimental classification target
model_df["large_increase"] = (
    model_df["glucose_future_2h"] - model_df["glucose"] >= 100
).astype(int)

# 2. Reuse the features from Random Forest V2
features = [
    "glucose",
    "calories",
    "heart_rate",
    "steps",
    "basal_rate",
    "bolus_volume_delivered",
    "carb_input",
    "glucose_lag_1",
    "glucose_lag_3",
    "glucose_lag_6",
    "glucose_lag_12",
    "glucose_mean_3",
    "glucose_mean_6",
    "glucose_std_6",
    "glucose_change_15m",
    "glucose_change_30m"
]

TARGET_CLASS = "large_increase"

# 3. Reuse the existing participant split
train_mask = model_df["participant"].isin(train_participants)
test_mask = model_df["participant"].isin(test_participants)

X_train = model_df.loc[train_mask, features]
y_train = model_df.loc[train_mask, TARGET_CLASS]

X_test = model_df.loc[test_mask, features]
y_test = model_df.loc[test_mask, TARGET_CLASS]

# 4. Train classifier
classifier = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

classifier.fit(X_train, y_train)

# 5. Predict event probabilities
probabilities = classifier.predict_proba(X_test)[:, 1]

# Use 0.5 as the initial threshold for evaluation
predictions = (probabilities >= 0.5).astype(int)

# 6. Evaluate
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

print("\nActual event counts:")
print(y_test.value_counts().sort_index())

print("\nConfusion matrix [ [TN, FP], [FN, TP] ]:")
print(confusion_matrix(y_test, predictions, labels=[0, 1]))

print("\nClassification report:")
print(classification_report(
    y_test,
    predictions,
    labels=[0, 1],
    target_names=["No large increase", "Large increase"],
    zero_division=0
))

print(
    "Average precision:",
    round(average_precision_score(y_test, probabilities), 4)
)

if y_test.nunique() == 2:
    print("ROC-AUC:", round(roc_auc_score(y_test, probabilities), 4))

In [ ]:
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score

threshold_results = []

for threshold in [0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80]:
    predictions_threshold = (
        probabilities >= threshold
    ).astype(int)

    precision = precision_score(
        y_test, predictions_threshold, zero_division=0
    )
    recall = recall_score(
        y_test, predictions_threshold, zero_division=0
    )
    f1 = f1_score(
        y_test, predictions_threshold, zero_division=0
    )

    threshold_results.append({
        "threshold": threshold,
        "predicted_events": int(predictions_threshold.sum()),
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "f1_score": round(f1, 4)
    })

threshold_df = pd.DataFrame(threshold_results)

print(threshold_df.to_string(index=False))

In [ ]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score
)
import pandas as pd

# Use the existing test participant IDs and predicted probabilities.
participant_results = []

test_participants_array = (
    model_df.loc[test_mask, "participant"].to_numpy()
)

actual_events = y_test.to_numpy()

for participant in sorted(set(test_participants_array)):
    mask = test_participants_array == participant

    y_actual = actual_events[mask]
    y_prob = probabilities[mask]
    y_pred = (y_prob >= 0.5).astype(int)

    participant_results.append({
        "participant": participant,
        "test_rows": int(mask.sum()),
        "actual_events": int(y_actual.sum()),
        "predicted_events": int(y_pred.sum()),
        "precision": round(
            precision_score(y_actual, y_pred, zero_division=0), 4
        ),
        "recall": round(
            recall_score(y_actual, y_pred, zero_division=0), 4
        ),
        "f1_score": round(
            f1_score(y_actual, y_pred, zero_division=0), 4
        ),
        "average_precision": (
            round(average_precision_score(y_actual, y_prob), 4)
            if y_actual.sum() > 0 else None
        )
    })

participant_results_df = pd.DataFrame(participant_results)

print(participant_results_df.to_string(index=False))

In [ ]:
# Compare glucose patterns across test participants

participant_data = model_df.loc[
    test_mask,
    ["participant", "glucose", "glucose_future_2h"]
].copy()

participant_data["large_increase"] = (
    participant_data["glucose_future_2h"]
    - participant_data["glucose"] >= 100
).astype(int)

participant_summary = (
    participant_data
    .groupby("participant")
    .agg(
        readings=("glucose", "size"),
        current_glucose_mean=("glucose", "mean"),
        current_glucose_std=("glucose", "std"),
        future_glucose_mean=("glucose_future_2h", "mean"),
        future_glucose_std=("glucose_future_2h", "std"),
        large_increase_events=("large_increase", "sum"),
        large_increase_rate=("large_increase", "mean")
    )
    .round(3)
)

participant_summary["large_increase_rate_pct"] = (
    participant_summary["large_increase_rate"] * 100
).round(2)

print(participant_summary.to_string())

In [ ]:
# Inspect the training and test features used by the classifier

print("FEATURE QUALITY REPORT")
print("=" * 60)

print("\nMissing values:")
print(model_df[features].isna().sum())

print("\nUnique values per feature:")
print(model_df[features].nunique())

print("\nFeature summary:")
print(model_df[features].describe().T.round(2))

print("\nClass distribution:")
print(
    model_df["large_increase"]
    .value_counts()
    .rename(index={0: "No large increase", 1: "Large increase"})
)

print("\nParticipant count:", model_df["participant"].nunique())

In [ ]:
import numpy as np
import pandas as pd

# 1. Inspect feature quality
quality_report = pd.DataFrame({
    "missing_count": model_df[features].isna().sum(),
    "missing_pct": (model_df[features].isna().mean() * 100).round(3),
    "unique_values": model_df[features].nunique(),
    "std": model_df[features].std().round(3),
    "min": model_df[features].min(),
    "median": model_df[features].median(),
    "max": model_df[features].max()
})

print("FEATURE QUALITY REPORT")
display(quality_report)

# 2. Check constant and completely missing features
print("\nConstant features:")
print(quality_report.index[quality_report["unique_values"] <= 1].tolist())

print("\nCompletely missing features:")
print(quality_report.index[quality_report["missing_pct"] == 100].tolist())

# 3. Check infinite values
numeric_features = model_df[features].select_dtypes(include=np.number)

print("\nInfinite values per feature:")
print(
    pd.Series(
        np.isinf(numeric_features).sum(),
        name="infinite_count"
    )
)

# 4. Check target distribution
print("\nTARGET DISTRIBUTION")
print(model_df["large_increase"].value_counts())
print("\nTarget percentages:")
print(
    (model_df["large_increase"].value_counts(normalize=True) * 100)
    .round(2)
)



In [ ]:
# Task 1B: Investigate suspicious observations

checks = {
    "glucose_above_400": model_df["glucose"] > 400,
    "glucose_below_40": model_df["glucose"] < 40,
    "negative_bolus": model_df["bolus_volume_delivered"] < 0,
    "abs_change_15m_over_100": model_df["glucose_change_15m"].abs() > 100,
    "abs_change_30m_over_150": model_df["glucose_change_30m"].abs() > 150,
}

print("SUSPICIOUS VALUE COUNTS")
for name, mask in checks.items():
    print(f"{name}: {mask.sum()}")

# Display sample rows for investigation
for name in [
    "negative_bolus",
    "abs_change_15m_over_100",
    "abs_change_30m_over_150"
]:
    mask = checks[name]

    print(f"\n{name} — sample rows")
    display(
        model_df.loc[
            mask,
            ["participant", "time", "glucose",
             "bolus_volume_delivered",
             "glucose_change_15m",
             "glucose_change_30m"]
        ].head(5)
    )

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

# Define feature groups
feature_groups = {
    "Current glucose only": [
        "glucose"
    ],

    "Glucose history": [
        "glucose",
        "glucose_lag_1",
        "glucose_lag_3",
        "glucose_lag_6",
        "glucose_lag_12",
        "glucose_mean_3",
        "glucose_mean_6",
        "glucose_std_6",
        "glucose_change_15m",
        "glucose_change_30m"
    ],

    "Wearable and treatment signals": [
        "calories",
        "heart_rate",
        "steps",
        "basal_rate",
        "bolus_volume_delivered",
        "carb_input"
    ],

    "Combined features": features
}

# Reuse the existing participant-level split
train_mask = model_df["participant"].isin(train_participants)
test_mask = model_df["participant"].isin(test_participants)

X_train_all = model_df.loc[train_mask]
X_test_all = model_df.loc[test_mask]

y_train_reg = X_train_all["glucose_future_2h"]
y_test_reg = X_test_all["glucose_future_2h"]

results = []

for group_name, group_features in feature_groups.items():

    model = RandomForestRegressor(
        n_estimators=100,
        max_depth=15,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train_all[group_features], y_train_reg)

    predictions = model.predict(X_test_all[group_features])

    results.append({
        "feature_group": group_name,
        "feature_count": len(group_features),
        "MAE": round(mean_absolute_error(y_test_reg, predictions), 3),
        "RMSE": round(
            np.sqrt(mean_squared_error(y_test_reg, predictions)), 3
        ),
        "R2": round(r2_score(y_test_reg, predictions), 4)
    })

results_df = pd.DataFrame(results)

print(results_df.to_string(index=False))

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import pandas as pd

glucose_features = [
    "glucose",
    "glucose_lag_1",
    "glucose_lag_3",
    "glucose_lag_6",
    "glucose_lag_12",
    "glucose_mean_3",
    "glucose_mean_6",
    "glucose_std_6",
    "glucose_change_15m",
    "glucose_change_30m"
]

additional_features = [
    "calories",
    "heart_rate",
    "steps",
    "basal_rate",
    "bolus_volume_delivered",
    "carb_input"
]

results = []

for extra_feature in ["None"] + additional_features:
    selected_features = (
        glucose_features
        if extra_feature == "None"
        else glucose_features + [extra_feature]
    )

    model = RandomForestRegressor(
        n_estimators=100,
        max_depth=15,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train_all[selected_features],
        y_train_reg
    )

    predictions = model.predict(
        X_test_all[selected_features]
    )

    results.append({
        "added_feature": extra_feature,
        "feature_count": len(selected_features),
        "MAE": round(
            mean_absolute_error(y_test_reg, predictions), 3
        ),
        "RMSE": round(
            np.sqrt(mean_squared_error(y_test_reg, predictions)), 3
        )
    })

ablation_results = (
    pd.DataFrame(results)
    .sort_values("MAE")
    .reset_index(drop=True)
)

print(ablation_results.to_string(index=False))

In [ ]:
import numpy as np
import pandas as pd

from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    HistGradientBoostingRegressor
)
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Selected features based on our experiments
selected_features = [
    "glucose",
    "glucose_lag_1",
    "glucose_lag_3",
    "glucose_lag_6",
    "glucose_lag_12",
    "glucose_mean_3",
    "glucose_mean_6",
    "glucose_std_6",
    "glucose_change_15m",
    "glucose_change_30m",
    "steps",
    "calories"
]

X_train = X_train_all[selected_features]
X_test = X_test_all[selected_features]

models = {
    "Random Forest": RandomForestRegressor(
        n_estimators=150,
        max_depth=15,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    ),

    "Extra Trees": ExtraTreesRegressor(
        n_estimators=150,
        max_depth=15,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    ),

    "HistGradientBoosting": HistGradientBoostingRegressor(
        max_iter=150,
        max_leaf_nodes=31,
        learning_rate=0.1,
        l2_regularization=1.0,
        random_state=42
    )
}

model_results = []

for name, model in models.items():
    model.fit(X_train, y_train_reg)
    predictions = model.predict(X_test)

    model_results.append({
        "model": name,
        "MAE": round(mean_absolute_error(y_test_reg, predictions), 3),
        "RMSE": round(
            np.sqrt(mean_squared_error(y_test_reg, predictions)), 3
        ),
        "R2": round(r2_score(y_test_reg, predictions), 4)
    })

model_comparison = (
    pd.DataFrame(model_results)
    .sort_values("MAE")
    .reset_index(drop=True)
)

print(model_comparison.to_string(index=False))

In [ ]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# 1. Refit the selected model using the existing split
best_model = HistGradientBoostingRegressor(
    max_iter=150,
    max_leaf_nodes=31,
    learning_rate=0.1,
    l2_regularization=1.0,
    random_state=42
)

best_model.fit(X_train, y_train_reg)
best_predictions = best_model.predict(X_test)

# 2. Align test targets, participant IDs, and baseline
actual = y_test_reg.to_numpy()
participant_ids = X_test_all["participant"].to_numpy()
baseline_predictions = X_test_all["glucose"].to_numpy()

# 3. Validate alignment
assert len(actual) == len(best_predictions) == len(participant_ids)
assert len(actual) == len(baseline_predictions)

print("Test rows:", len(actual))
print("Test participants:", len(np.unique(participant_ids)))

# 4. Overall evaluation
evaluation_results = []

for name, predictions in [
    ("Persistence Baseline", baseline_predictions),
    ("HistGradientBoosting", best_predictions)
]:
    evaluation_results.append({
        "Model": name,
        "MAE": round(mean_absolute_error(actual, predictions), 3),
        "RMSE": round(
            np.sqrt(mean_squared_error(actual, predictions)), 3
        ),
        "R2": round(r2_score(actual, predictions), 4)
    })

evaluation_df_final = pd.DataFrame(evaluation_results)

print("\nOVERALL EVALUATION")
print(evaluation_df_final.to_string(index=False))

# 5. Participant-level evaluation
participant_results = []

for participant in sorted(np.unique(participant_ids)):
    mask = participant_ids == participant

    participant_actual = actual[mask]
    participant_predictions = best_predictions[mask]
    participant_baseline = baseline_predictions[mask]

    participant_results.append({
        "Participant": participant,
        "Test_Rows": int(mask.sum()),
        "Model_MAE": round(
            mean_absolute_error(
                participant_actual, participant_predictions
            ), 3
        ),
        "Baseline_MAE": round(
            mean_absolute_error(
                participant_actual, participant_baseline
            ), 3
        ),
        "Model_RMSE": round(
            np.sqrt(mean_squared_error(
                participant_actual, participant_predictions
            )), 3
        )
    })

participant_results_df = pd.DataFrame(participant_results)

print("\nPARTICIPANT-LEVEL EVALUATION")
print(participant_results_df.to_string(index=False))

print(
    "\nMean participant MAE:",
    round(participant_results_df["Model_MAE"].mean(), 3)
)

print(
    "Mean participant RMSE:",
    round(participant_results_df["Model_RMSE"].mean(), 3)
)

print(
    "Participants where model beat baseline:",
    int(
        (
            participant_results_df["Model_MAE"]
            < participant_results_df["Baseline_MAE"]
        ).sum()
    ),
    "/",
    len(participant_results_df)
)
# 6. Save evaluation reports
from pathlib import Path

results_dir = Path("reports")
results_dir.mkdir(exist_ok=True)

evaluation_df_final.to_csv(
    results_dir / "overall_model_evaluation.csv",
    index=False
)

participant_results_df.to_csv(
    results_dir / "participant_model_evaluation.csv",
    index=False
)

print("\nEvaluation reports saved successfully!")
print("Saved:", results_dir / "overall_model_evaluation.csv")
print("Saved:", results_dir / "participant_model_evaluation.csv")


In [ ]:
import numpy as np
import pandas as pd

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Actual future glucose
actual = y_test_reg.to_numpy()

# Model prediction
model_pred = best_predictions

# Persistence baseline: future glucose = current glucose
baseline_pred = X_test_all["glucose"].to_numpy()

evaluation_results = []

for name, prediction in [
    ("Persistence baseline", baseline_pred),
    ("HistGradientBoosting", model_pred)
]:
    evaluation_results.append({
        "model": name,
        "MAE": round(mean_absolute_error(actual, prediction), 3),
        "RMSE": round(
            np.sqrt(mean_squared_error(actual, prediction)), 3
        ),
        "R2": round(r2_score(actual, prediction), 4)
    })

evaluation_df_final = pd.DataFrame(evaluation_results)

print(evaluation_df_final.to_string(index=False))

baseline_mae = mean_absolute_error(actual, baseline_pred)
model_mae = mean_absolute_error(actual, model_pred)

print("\nMAE improvement (%):",
      round((baseline_mae - model_mae) / baseline_mae * 100, 2))

In [ ]:
import pandas as pd
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error

# Evaluate feature importance using the held-out test set
importance_result = permutation_importance(
    best_model,
    X_test,
    y_test_reg,
    scoring="neg_mean_absolute_error",
    n_repeats=3,
    random_state=42,
    n_jobs=-1
)

importance_df = pd.DataFrame({
    "feature": X_test.columns,
    "importance_mean": importance_result.importances_mean,
    "importance_std": importance_result.importances_std
})

importance_df = importance_df.sort_values(
    "importance_mean",
    ascending=False
).reset_index(drop=True)

print("TOP FEATURES — PERMUTATION IMPORTANCE")
print(importance_df.round(4).to_string(index=False))

In [ ]:
import pandas as pd

# 1. Load EHR patients
patients = ehr_tables["patients"].copy()

# 2. Inspect identifiers
print("=" * 60)
print("EHR PATIENT IDENTIFIERS")
print("=" * 60)

print("Total EHR patients:", len(patients))
print("Unique EHR patient IDs:", patients["Id"].nunique())
print("\nSample EHR patient IDs:")
print(patients["Id"].head(10).to_list())

# 3. Inspect CGM participant identifiers
print("\n" + "=" * 60)
print("CGM PARTICIPANT IDENTIFIERS")
print("=" * 60)

cgm_participants = sorted(
    prediction_data["participant"].dropna().unique()
)

print("Total CGM participants:", len(cgm_participants))
print("\nCGM participant IDs:")
print(cgm_participants)

# 4. Check exact identifier overlap
ehr_ids = set(patients["Id"].dropna().astype(str))
cgm_ids = set(prediction_data["participant"].dropna().astype(str))

exact_matches = ehr_ids.intersection(cgm_ids)

print("\n" + "=" * 60)
print("IDENTIFIER MATCHING RESULTS")
print("=" * 60)

print("Exact identifier matches:", len(exact_matches))
print("EHR patients:", len(ehr_ids))
print("CGM participants:", len(cgm_ids))

if exact_matches:
    print("\nMatching identifier values:")
    print(sorted(exact_matches)[:20])
else:
    print("\nNo exact identifier matches found.")

# 5. Inspect available timestamp columns
print("\nEHR patient columns:")
print(patients.columns.tolist())

print("\nCGM columns:")
print(prediction_data.columns.tolist())

In [ ]:
from pathlib import Path

PROJECT_DIR = Path.cwd()

print("Project directory:", PROJECT_DIR)

keywords = [
    "mapping", "match", "linkage", "patient",
    "participant", "readme", "metadata", "dictionary"
]

matches = []

for path in PROJECT_DIR.rglob("*"):
    if not path.is_file():
        continue

    # Ignore virtual environments and dependency folders
    if any(part in {".venv", ".venv313", "node_modules", ".git"}
           for part in path.parts):
        continue

    if any(word in path.name.lower() for word in keywords):
        matches.append(path.relative_to(PROJECT_DIR))

print("\nPotential mapping/documentation files:")

if matches:
    for path in matches[:100]:
        print(path)
else:
    print("No matching filenames found.")

In [ ]:
from pathlib import Path

PROJECT_DIR = Path.cwd().parent  # notebooks -> project root

print("Project root:", PROJECT_DIR)

keywords = [
    "mapping", "match", "linkage", "patient",
    "participant", "readme", "metadata", "dictionary",
    "data_description", "codebook"
]

matches = []

for path in PROJECT_DIR.rglob("*"):
    if not path.is_file():
        continue

    if any(
        part.lower() in {
            ".venv", ".venv313", "node_modules", ".git",
            "__pycache__"
        }
        for part in path.parts
    ):
        continue

    if any(word in path.name.lower() for word in keywords):
        matches.append(path.relative_to(PROJECT_DIR))

print("\nPotential mapping/documentation files:")

if matches:
    for path in matches:
        print(path)
else:
    print("No matching filenames found.")

In [ ]:
from pathlib import Path

PROJECT_DIR = Path.cwd().parent

# 1. Read README
readme_path = PROJECT_DIR / "README.md"

if readme_path.exists():
    print("=" * 70)
    print("PROJECT README")
    print("=" * 70)
    print(readme_path.read_text(encoding="utf-8")[:12000])
else:
    print("README.md not found")

# 2. Inspect top-level project files and data directories
print("\n" + "=" * 70)
print("PROJECT STRUCTURE")
print("=" * 70)

for folder in ["data", "data/ehr", "data/cgm", "data/wearables", "docs"]:
    path = PROJECT_DIR / folder
    if path.exists():
        print(f"\n{folder}/")
        for item in sorted(path.iterdir()):
            print("  ", item.name)

# 3. Inspect patient identifiers and available metadata
import pandas as pd

patients_path = PROJECT_DIR / "data" / "ehr" / "patients.csv"
patients = pd.read_csv(patients_path, dtype={"Id": "string"})

print("\n" + "=" * 70)
print("EHR PATIENT METADATA")
print("=" * 70)
print("Shape:", patients.shape)
print("Columns:", patients.columns.tolist())
print("\nFirst 3 rows (identifying columns excluded):")

safe_columns = [
    col for col in patients.columns
    if col.upper() not in {
        "SSN", "DRIVERS", "PASSPORT", "FIRST", "MIDDLE",
        "LAST", "MAIDEN", "ADDRESS", "ZIP"
    }
]
display(patients[safe_columns].head(3))

In [ ]:
from pathlib import Path

PROJECT_DIR = Path.cwd().parent

print("=" * 65)
print("1. ALL PROJECT DOCUMENTATION FILES")
print("=" * 65)

for path in PROJECT_DIR.rglob("*"):
    if not path.is_file():
        continue

    if any(part.lower() in {
        ".git", ".venv", ".venv313", "node_modules", "__pycache__"
    } for part in path.parts):
        continue

    if path.suffix.lower() in {
        ".md", ".txt", ".json", ".yaml", ".yml", ".pdf"
    }:
        print(path.relative_to(PROJECT_DIR))

print("\n" + "=" * 65)
print("2. WEARABLE DATASET STRUCTURE")
print("=" * 65)

wearable_dir = PROJECT_DIR / "data" / "wearable"

if wearable_dir.exists():
    for path in sorted(wearable_dir.rglob("*")):
        if path.is_file():
            print(
                f"{path.relative_to(PROJECT_DIR)} "
                f"({path.stat().st_size:,} bytes)"
            )
else:
    print("Wearable directory not found")

print("\n" + "=" * 65)
print("3. SEARCH DOCUMENTATION FOR LINKAGE TERMS")
print("=" * 65)

keywords = [
    "participant", "mapping", "crosswalk", "patient id",
    "ehr", "cgm", "cohort", "identifier", "join"
]

for path in PROJECT_DIR.rglob("*"):
    if not path.is_file() or path.suffix.lower() not in {
        ".md", ".txt", ".json", ".yaml", ".yml"
    }:
        continue

    if any(part.lower() in {
        ".git", ".venv", ".venv313", "node_modules", "__pycache__"
    } for part in path.parts):
        continue

    try:
        text = path.read_text(encoding="utf-8", errors="ignore")
        hits = [
            line.strip() for line in text.splitlines()
            if any(word in line.lower() for word in keywords)
        ]

        if hits:
            print(f"\nFile: {path.relative_to(PROJECT_DIR)}")
            for line in hits[:10]:
                print(" ", line[:250])
    except OSError:
        pass

In [ ]:
from pathlib import Path

PROJECT_DIR = Path.cwd().parent

# 1. Read README
readme = PROJECT_DIR / "README.md"

print("=" * 60)
print("README.md")
print("=" * 60)

if readme.exists():
    print(repr(readme.read_text(encoding="utf-8", errors="replace")[:15000]))

# 2. Read requirements
requirements = PROJECT_DIR / "requirements.txt"

print("\n" + "=" * 60)
print("requirements.txt")
print("=" * 60)

if requirements.exists():
    print(requirements.read_text(encoding="utf-8", errors="replace"))

# 3. Inspect wearable dataset files
wearable_dir = (
    PROJECT_DIR
    / "data"
    / "wearable"
    / "HUPA-UCM"
    / "HUPA-UCM Diabetes Dataset"
)

print("\n" + "=" * 60)
print("DATASET ROOT CONTENTS")
print("=" * 60)

if wearable_dir.exists():
    for item in sorted(wearable_dir.iterdir()):
        print(f"{item.name}  {'[DIR]' if item.is_dir() else '[FILE]'}")
else:
    print("Dataset root not found:", wearable_dir)

# 4. Look for dataset metadata files
print("\n" + "=" * 60)
print("POTENTIAL METADATA FILES")
print("=" * 60)

if wearable_dir.exists():
    for path in wearable_dir.rglob("*"):
        if path.is_file() and (
            path.suffix.lower() in {".pdf", ".txt", ".md", ".json", ".xlsx"}
            or any(word in path.name.lower() for word in [
                "readme", "metadata", "description", "dictionary", "license"
            ])
        ):
            print(path.relative_to(PROJECT_DIR))

In [ ]:
from pathlib import Path
import subprocess

PROJECT_DIR = Path.cwd().parent

dataset_root = (
    PROJECT_DIR
    / "data"
    / "wearable"
    / "HUPA-UCM"
    / "HUPA-UCM Diabetes Dataset"
)

# 1. Inspect Raw_Data structure
print("=" * 60)
print("RAW DATASET STRUCTURE")
print("=" * 60)

raw_dir = dataset_root / "Raw_Data"

if raw_dir.exists():
    for item in sorted(raw_dir.rglob("*")):
        if item.is_file():
            print(item.relative_to(PROJECT_DIR))

# 2. Check Git-tracked documentation and mapping-related files
print("\n" + "=" * 60)
print("GIT-TRACKED FILES RELATED TO MAPPING/DOCUMENTATION")
print("=" * 60)

result = subprocess.run(
    ["git", "ls-files"],
    cwd=PROJECT_DIR,
    capture_output=True,
    text=True
)

terms = [
    "mapping", "crosswalk", "metadata", "readme",
    "dictionary", "linkage", "participant", "patient"
]

for line in result.stdout.splitlines():
    if any(term in line.lower() for term in terms):
        print(line)

# 3. Inspect Git remotes and recent commits for project context
print("\n" + "=" * 60)
print("GIT REMOTE")
print("=" * 60)

remote = subprocess.run(
    ["git", "remote", "-v"],
    cwd=PROJECT_DIR,
    capture_output=True,
    text=True
)
print(remote.stdout or "No Git remote found")

print("\n" + "=" * 60)
print("RECENT COMMITS")
print("=" * 60)

log = subprocess.run(
    ["git", "log", "-5", "--oneline"],
    cwd=PROJECT_DIR,
    capture_output=True,
    text=True
)
print(log.stdout or "No commits found")

In [ ]:
from pathlib import Path

PROJECT_DIR = Path.cwd().parent

print("PROJECT ROOT:", PROJECT_DIR)

print("\nPotential saved models:")
model_extensions = {".pkl", ".joblib", ".onnx", ".pt", ".pth"}

model_files = [
    path for path in PROJECT_DIR.rglob("*")
    if path.is_file()
    and path.suffix.lower() in model_extensions
    and not any(
        part.lower() in {
            ".git", ".venv", ".venv313", "node_modules", "__pycache__"
        }
        for part in path.parts
    )
]

if model_files:
    for path in model_files:
        print(path.relative_to(PROJECT_DIR))
else:
    print("No saved model files found.")

print("\nExisting application-related directories:")
for name in ["app", "dashboard", "frontend", "src", "models"]:
    path = PROJECT_DIR / name
    if path.exists():
        print(f"{name}/ exists")

In [ ]:
from pathlib import Path
import joblib
import json

# Project root
PROJECT_DIR = Path.cwd().parent

# Create model directory
MODEL_DIR = PROJECT_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Verify that the trained model exists
assert "best_model" in globals(), (
    "best_model not found. Run the model-training cell first."
)

# Save trained model
model_path = MODEL_DIR / "glucose_forecaster.joblib"
joblib.dump(best_model, model_path)

# Save the exact feature configuration
feature_config = {
    "model_name": "HistGradientBoostingRegressor",
    "target": "glucose_future_2h",
    "prediction_horizon_minutes": 120,
    "features": X_train.columns.tolist(),
    "evaluation": {
        "mae_mg_dl": 30.234,
        "rmse_mg_dl": 39.879,
        "r2": 0.3583
    },
    "status": "experimental_prototype",
    "clinical_use": False
}

config_path = MODEL_DIR / "glucose_forecaster_config.json"

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(feature_config, f, indent=4)

print("Model saved:", model_path)
print("Config saved:", config_path)
print("Features:", len(feature_config["features"]))
print("Model size:", f"{model_path.stat().st_size / 1024:.2f} KB")

In [ ]:
import joblib
import json

loaded_model = joblib.load(model_path)

with open(config_path, encoding="utf-8") as f:
    loaded_config = json.load(f)

print("Model loaded:", type(loaded_model).__name__)
print("Prediction horizon:", loaded_config["prediction_horizon_minutes"], "minutes")
print("Feature count:", len(loaded_config["features"]))
print("Features:", loaded_config["features"])

In [ ]:
from pathlib import Path

PROJECT_DIR = Path.cwd().parent
DASHBOARD_DIR = PROJECT_DIR / "dashboard"

print("DASHBOARD DIRECTORY:", DASHBOARD_DIR)
print("=" * 60)

if not DASHBOARD_DIR.exists():
    print("Dashboard directory does not exist.")
else:
    for path in sorted(DASHBOARD_DIR.rglob("*")):
        if not path.is_file():
            continue

        # Avoid listing generated/dependency files
        if any(part.lower() in {
            "node_modules", ".git", "__pycache__", ".venv", ".venv313"
        } for part in path.parts):
            continue

        print(f"{path.relative_to(PROJECT_DIR)} ({path.stat().st_size:,} bytes)")

# 
Verify Exact Feature Engineering

In [ ]:
import numpy as np
import pandas as pd

# --------------------------------------------------
# 1. Verify that the required variables exist
# --------------------------------------------------

required_variables = [
    "df",
    "model_df",
    "prediction_data",
    "TARGET",
]

missing_variables = [
    name for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise NameError(
        f"Run the notebook cells that create these variables first: "
        f"{missing_variables}"
    )

# --------------------------------------------------
# 2. Define the expected model features
# --------------------------------------------------

FEATURES = [
    "glucose",
    "glucose_lag_1",
    "glucose_lag_3",
    "glucose_lag_6",
    "glucose_lag_12",
    "glucose_mean_3",
    "glucose_mean_6",
    "glucose_std_6",
    "glucose_change_15m",
    "glucose_change_30m",
    "steps",
    "calories",
]

print("=" * 65)
print("STEP 1: FEATURE ENGINEERING AUDIT")
print("=" * 65)

print("Original engineered dataset:", df.shape)
print("Model-ready dataset:", model_df.shape)
print("Expected model features:", len(FEATURES))

# --------------------------------------------------
# 3. Check required columns
# --------------------------------------------------

required_columns = [
    "participant",
    "time",
    "glucose",
    "steps",
    "calories",
    TARGET,
] + FEATURES

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

print("\nMissing required columns:", missing_columns)

if missing_columns:
    raise ValueError(
        "Some required columns are missing. "
        "Check the feature-engineering cells."
    )

# --------------------------------------------------
# 4. Check chronological ordering
# --------------------------------------------------

ordered = df.sort_values(
    ["participant", "time"]
).reset_index(drop=True)

already_sorted = (
    df[["participant", "time"]]
    .reset_index(drop=True)
    .equals(ordered[["participant", "time"]])
)

print("\nRows already sorted by participant and time:", already_sorted)

# --------------------------------------------------
# 5. Independently recalculate historical features
# --------------------------------------------------

audit = ordered[
    ["participant", "time", "glucose", "steps", "calories", TARGET]
].copy()

grouped_glucose = audit.groupby(
    "participant",
    sort=False
)["glucose"]

# Historical lags
for lag in [1, 3, 6, 12]:
    audit[f"glucose_lag_{lag}"] = grouped_glucose.shift(lag)

# Historical rolling statistics
audit["glucose_mean_3"] = (
    grouped_glucose
    .transform(lambda s: s.shift(1).rolling(3).mean())
)

audit["glucose_mean_6"] = (
    grouped_glucose
    .transform(lambda s: s.shift(1).rolling(6).mean())
)

audit["glucose_std_6"] = (
    grouped_glucose
    .transform(lambda s: s.shift(1).rolling(6).std())
)

# Glucose changes
audit["glucose_change_15m"] = (
    audit["glucose"] - audit["glucose_lag_3"]
)

audit["glucose_change_30m"] = (
    audit["glucose"] - audit["glucose_lag_6"]
)

# --------------------------------------------------
# 6. Compare existing and independently recalculated features
# --------------------------------------------------

comparison = []

for feature in FEATURES:
    if feature in ["glucose", "steps", "calories"]:
        # These are source measurements, not engineered features.
        expected = ordered[feature].reset_index(drop=True)
        actual = audit[feature].reset_index(drop=True)
    else:
        expected = audit[feature].reset_index(drop=True)
        actual = ordered[feature].reset_index(drop=True)

    valid = expected.notna() & actual.notna()

    if valid.sum() == 0:
        max_difference = np.nan
        matches = False
    else:
        differences = (
            expected[valid].astype(float)
            - actual[valid].astype(float)
        ).abs()

        max_difference = differences.max()
        matches = np.allclose(
            expected[valid].astype(float),
            actual[valid].astype(float),
            rtol=1e-7,
            atol=1e-7,
        )

    comparison.append({
        "feature": feature,
        "existing_missing": int(actual.isna().sum()),
        "recalculated_missing": int(expected.isna().sum()),
        "max_absolute_difference": max_difference,
        "matches": matches,
    })

comparison_df = pd.DataFrame(comparison)

print("\nFEATURE COMPARISON")
display(comparison_df)

# --------------------------------------------------
# 7. Check whether the model-ready data has missing values
# --------------------------------------------------

print("\nMODEL-READY FEATURE QUALITY")

model_feature_missing = model_df[FEATURES].isna().sum()

display(
    model_feature_missing.rename("missing_values").to_frame()
)

print("\nAll 12 model features present:",
      all(feature in model_df.columns for feature in FEATURES))

print("Missing feature values in model_df:",
      int(model_df[FEATURES].isna().sum().sum()))

print("\nAudit completed. No source data was modified.")

In [ ]:
import numpy as np
import pandas as pd

print("=" * 65)
print("STEP 2: TWO-HOUR TARGET ALIGNMENT AUDIT")
print("=" * 65)

# Sort records within each participant
check_df = prediction_data.sort_values(
    ["participant", "time"]
).reset_index(drop=True)

# Independently reconstruct the future target
check_df["expected_future_glucose"] = (
    check_df.groupby("participant", sort=False)["glucose"]
    .shift(-24)
)

# Compare the existing target with the reconstructed target
valid = (
    check_df["glucose_future_2h"].notna()
    & check_df["expected_future_glucose"].notna()
)

difference = (
    check_df.loc[valid, "glucose_future_2h"]
    - check_df.loc[valid, "expected_future_glucose"]
).abs()

print("Total prediction rows:", len(check_df))
print("Rows compared:", int(valid.sum()))
print("Rows without a valid 24-step future target:", int((~valid).sum()))
print("Maximum target difference:", difference.max())
print("All target values match:", np.allclose(
    check_df.loc[valid, "glucose_future_2h"],
    check_df.loc[valid, "expected_future_glucose"],
    rtol=0,
    atol=1e-9
))

# Check that the target is approximately 120 minutes ahead
check_df["future_time"] = (
    check_df.groupby("participant", sort=False)["time"]
    .shift(-24)
)

check_df["actual_horizon_minutes"] = (
    check_df["future_time"] - check_df["time"]
).dt.total_seconds() / 60

horizon = check_df["actual_horizon_minutes"].dropna()

print("\nACTUAL PREDICTION HORIZON")
print("Minimum:", horizon.min(), "minutes")
print("Median:", horizon.median(), "minutes")
print("Maximum:", horizon.max(), "minutes")
print("All horizons equal 120 minutes:", (horizon == 120).all())

# Confirm that the target shift stays within each participant
print("\nParticipant count:", check_df["participant"].nunique())
print("Participants with valid future targets:",
      check_df.loc[valid, "participant"].nunique())

In [ ]:
import numpy as np
import pandas as pd

print("=" * 65)
print("STEP 3: DATA LEAKAGE AUDIT")
print("=" * 65)

# --------------------------------------------------
# 1. Confirm target is not part of the model features
# --------------------------------------------------

feature_config = [
    "glucose",
    "glucose_lag_1",
    "glucose_lag_3",
    "glucose_lag_6",
    "glucose_lag_12",
    "glucose_mean_3",
    "glucose_mean_6",
    "glucose_std_6",
    "glucose_change_15m",
    "glucose_change_30m",
    "steps",
    "calories",
]

target_column = "glucose_future_2h"

print("Target included in feature list:",
      target_column in feature_config)

# --------------------------------------------------
# 2. Verify rolling features use previous readings only
# --------------------------------------------------

leakage_check = (
    df.sort_values(["participant", "time"])
    .reset_index(drop=True)
    .copy()
)

historical_group = leakage_check.groupby(
    "participant",
    sort=False
)["glucose"]

# Correct historical rolling calculations
leakage_check["expected_mean_3"] = (
    historical_group
    .transform(lambda s: s.shift(1).rolling(3).mean())
)

leakage_check["expected_mean_6"] = (
    historical_group
    .transform(lambda s: s.shift(1).rolling(6).mean())
)

leakage_check["expected_std_6"] = (
    historical_group
    .transform(lambda s: s.shift(1).rolling(6).std())
)

for actual, expected in [
    ("glucose_mean_3", "expected_mean_3"),
    ("glucose_mean_6", "expected_mean_6"),
    ("glucose_std_6", "expected_std_6"),
]:
    valid = (
        leakage_check[actual].notna()
        & leakage_check[expected].notna()
    )

    matches = np.allclose(
        leakage_check.loc[valid, actual],
        leakage_check.loc[valid, expected],
        rtol=1e-7,
        atol=1e-7,
    )

    print(f"{actual} uses historical readings only: {matches}")

# --------------------------------------------------
# 3. Verify lag features stay within each participant
# --------------------------------------------------

cross_participant_leakage = []

for lag in [1, 3, 6, 12]:
    expected_lag = (
        leakage_check.groupby("participant", sort=False)["glucose"]
        .shift(lag)
    )

    valid = (
        leakage_check[f"glucose_lag_{lag}"].notna()
        & expected_lag.notna()
    )

    matches = np.allclose(
        leakage_check.loc[valid, f"glucose_lag_{lag}"],
        expected_lag.loc[valid],
        rtol=0,
        atol=0,
    )

    cross_participant_leakage.append(matches)

    print(f"glucose_lag_{lag} stays within participant: {matches}")

# --------------------------------------------------
# 4. Check final model-ready input columns
# --------------------------------------------------

actual_model_features = [
    col for col in feature_config
    if col in model_df.columns
]

print("\nExpected features present:",
      len(actual_model_features) == len(feature_config))

print("Number of model features:", len(actual_model_features))

print("Target column in model features:",
      target_column in actual_model_features)

print("\nNo source data was modified.")

In [ ]:
import numpy as np
import joblib
from pathlib import Path

print("=" * 65)
print("STEP 4: SAVED MODEL REPRODUCIBILITY AUDIT")
print("=" * 65)

# 1. Verify the saved model path
PROJECT_DIR = Path.cwd().parent
MODEL_PATH = PROJECT_DIR / "models" / "glucose_forecaster.joblib"

assert MODEL_PATH.exists(), f"Model not found: {MODEL_PATH}"

# 2. Load the saved model
saved_model = joblib.load(MODEL_PATH)

# 3. Confirm feature order from the saved configuration
feature_config = [
    "glucose",
    "glucose_lag_1",
    "glucose_lag_3",
    "glucose_lag_6",
    "glucose_lag_12",
    "glucose_mean_3",
    "glucose_mean_6",
    "glucose_std_6",
    "glucose_change_15m",
    "glucose_change_30m",
    "steps",
    "calories",
]

# 4. Select a small sample of model-ready rows
sample = model_df[feature_config].head(100).copy()

assert not sample.isna().any().any(), (
    "Sample contains missing feature values."
)

# 5. Generate predictions using the original and saved models
original_predictions = best_model.predict(sample)
saved_predictions = saved_model.predict(sample)

# 6. Compare results
differences = np.abs(
    original_predictions - saved_predictions
)

print("Sample rows:", len(sample))
print("Maximum absolute prediction difference:", differences.max())
print(
    "Predictions match:",
    np.allclose(
        original_predictions,
        saved_predictions,
        rtol=1e-10,
        atol=1e-10,
    ),
)

print("\nFirst five comparisons:")

for i in range(min(5, len(sample))):
    print(
        f"Row {i}: "
        f"Original={original_predictions[i]:.6f}, "
        f"Saved={saved_predictions[i]:.6f}, "
        f"Difference={differences[i]:.10f}"
    )

# Build the prediction service 


In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path.cwd().parent

CGM_DIR = (
    PROJECT_DIR
    / "data"
    / "wearable"
    / "HUPA-UCM"
    / "HUPA-UCM Diabetes Dataset"
    / "Preprocessed"
)

print("=" * 65)
print("CGM DATA SOURCE CHECK")
print("=" * 65)

print("CGM directory exists:", CGM_DIR.exists())

csv_files = sorted(CGM_DIR.glob("*.csv"))

print("Participant CSV files:", len(csv_files))

if not csv_files:
    raise FileNotFoundError(f"No participant CSV files found in {CGM_DIR}")

# Inspect one participant file
sample_file = csv_files[0]
sample_raw = pd.read_csv(sample_file, sep=";")

print("\nSample file:", sample_file.name)
print("Shape:", sample_raw.shape)
print("Columns:", sample_raw.columns.tolist())

print("\nData types:")
print(sample_raw.dtypes)

print("\nFirst five rows:")
display(sample_raw.head())

print("\nMissing values:")
display(sample_raw.isna().sum().to_frame("missing_values"))

In [ ]:
from pathlib import Path

print("Notebook working directory:")
print(Path.cwd())

print("\nOverall report location:")
print(Path("reports/overall_model_evaluation.csv").resolve())

print("\nParticipant report location:")
print(Path("reports/participant_model_evaluation.csv").resolve())

print("\nFiles exist?")
print(Path("reports/overall_model_evaluation.csv").exists())
print(Path("reports/participant_model_evaluation.csv").exists())

In [ ]:
from pathlib import Path
import shutil

project_dir = Path.cwd().parent
source_dir = Path.cwd() / "reports"
target_dir = project_dir / "reports"

target_dir.mkdir(parents=True, exist_ok=True)

for filename in [
    "overall_model_evaluation.csv",
    "participant_model_evaluation.csv"
]:
    shutil.copy2(source_dir / filename, target_dir / filename)

print("Reports copied to:", target_dir)
print("Files:", [f.name for f in target_dir.glob("*.csv")])